# 台股基本面與估值版：依序執行至批次匯出即可。
# 每次輸出 reports/詳細版 與 reports/簡化版 各一個 Excel；簡化版保留核心估值、評分與分析師共識欄位。
# 技術面／籌碼不參與本版分析；Historical PE 以 1Y／3Y／5Y 分期呈現，不納入基本面分數；僅與合理價共同形成獨立的價格投資建議。
# 歷史彙整僅定義函式，不會在 Run All 時額外輸出或混讀兩個版本。
# Yahoo例外：Forward EPS、分析師共識與目標價直接讀取；財務安全性不足時整組用Yahoo同年度財報；ROIC無法計算時以Yahoo補算，其餘仍FinMind。


In [ ]:
# 設立&基本組態
# =========================
# Setup & config
# =========================
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import date
from typing import Any, Iterable
import logging
import math
import pandas as pd
import yaml

from valuation_history import (
    assess_pe_valuation_applicability,
    calculate_fundamental_score,
    calculate_historical_pe_context,
    calculate_historical_ratio_windows,
    calculate_price_recommendation,
    select_historical_valuation_reference,
)

LOGGER = logging.getLogger(__name__)

# =========================
# Config
# =========================
def load_config(path: str | Path | None = None):

    config_path = (
        Path(path)
        if path
        else Path("config/TW.yaml")
    )


    with config_path.open(
        encoding="utf-8"
    ) as file:

        config = yaml.safe_load(file)


    # 加入設定檔名稱
    config["_config_name"] = (
        config_path.stem
    )
    return config


In [ ]:
# 共同應用的函式
# =========================
# Shared helpers
# =========================
def safe_divide(numerator: float | None, denominator: float | None) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    result = numerator / denominator
    return result if math.isfinite(result) else None


def percentile_rank(value: float | None, values: Iterable[float]) -> float | None:
    cleaned = sorted(item for item in values if item is not None and math.isfinite(item))
    if value is None or not cleaned:
        return None
    return 100 * sum(item <= value for item in cleaned) / len(cleaned)

In [ ]:
# 函式定義
# =========================
# Data Models
# =========================

# =========================
# Core / Profile Models
# =========================
@dataclass
class Classification:
    market: str
    sector: str

# =========================
# Financial Data Models
# =========================
@dataclass
class FinancialSnapshot:
    ticker: str
    source: str
    as_of: date
    asset_type: str | None = None
    current_price: float | None = None
    week_52_high: float | None = None
    week_52_low: float | None = None
    market_cap: float | None = None
    enterprise_value: float | None = None
    revenue: float | None = None
    gross_profit: float | None = None
    operating_income: float | None = None
    net_income: float | None = None
    eps: float | None = None
    ttm_eps: float | None = None
    eps_ttm_nowcast: float | None = None
    pe_ttm_nowcast: float | None = None
    forward_eps: float | None = None
    forward_eps_meta: dict[str, Any] = field(default_factory=dict)
    book_value: float | None = None
    book_value_per_share: float | None = None
    free_cash_flow: float | None = None
    operating_cash_flow: float | None = None
    capex: float | None = None
    total_debt: float | None = None
    cash: float | None = None
    shares_outstanding: float | None = None
    average_shares_12m: float | None = None
    total_equity: float | None = None
    current_assets: float | None = None
    current_liabilities: float | None = None
    inventory: float | None = None
    interest_expense: float | None = None
    financial_history: dict[str, list[float]] = field(default_factory=dict)
    quarterly_reference: dict[str, Any] = field(default_factory=dict)
    cashflow_quality: dict[str, Any] = field(default_factory=dict)
    fundamental_review_inputs: dict[str, Any] = field(default_factory=dict)
    cash_review_scope: dict[str, Any] = field(default_factory=dict)
    safety_fallback_meta: dict[str, Any] = field(default_factory=dict)
    roic_fallback: float | None = None
    roic_meta: dict[str, Any] = field(default_factory=dict)

# =========================
# Quality Models
# =========================         
        
@dataclass
class QualityMetrics:
    roe: float | None = None
    roic: float | None = None
    gross_margin: float | None = None
    operating_margin: float | None = None
    net_margin: float | None = None
    debt_to_equity: float | None = None
    interest_coverage: float | None = None
    current_ratio: float | None = None
    quick_ratio: float | None = None
    fcf_margin: float | None = None
    revenue_cagr_1y: float | None = None    
    revenue_cagr_3y: float | None = None
    revenue_cagr_5y: float | None = None
    revenue_cagr_10y: float | None = None
    eps_cagr_1y: float | None = None
    eps_cagr_3y: float | None = None
    eps_cagr_5y: float | None = None
    eps_cagr_10y: float | None = None
    fcf_cagr_1y: float | None = None
    fcf_cagr_3y: float | None = None
    fcf_cagr_5y: float | None = None
    fcf_cagr_10y: float | None = None
    revenue_growth_3m: float | None = None
    revenue_growth_6m: float | None = None
    revenue_growth_9m: float | None = None
    eps_growth_3m: float | None = None
    eps_growth_6m: float | None = None
    eps_growth_9m: float | None = None
    ocf_growth_3m: float | None = None
    ocf_growth_6m: float | None = None
    ocf_growth_9m: float | None = None
    growth_status: dict[str, str] = field(default_factory=dict)
    growth_periods: dict[str, str] = field(default_factory=dict)
        
@dataclass(frozen=True)
class Recommendation:
    label: str
    stars: str
    score: float | None
        
# =========================
# Valuation Models
# =========================        
@dataclass
class ValuationMetrics:
    pe: float | None = None
    forward_pe: float | None = None          # FinMind 收盤價 / Yahoo Forward EPS
    pb: float | None = None
    peg: float | None = None
    fcf_yield: float | None = None
    ev_ebit: float | None = None
    ev_ebitda: float | None = None
    dcf_value_per_share: float | None = None
    # ========= Historical PE/PB distribution =========
    historical_pe_window_stats: dict[str, Any] | None = None
    historical_pb_window_stats: dict[str, Any] | None = None
    historical_pe_median_1y: float | None = None
    historical_pe_percentile_1y: float | None = None
    historical_pe_median_3y: float | None = None
    historical_pe_percentile_3y: float | None = None
    historical_pe_median_5y: float | None = None
    historical_pe_percentile_5y: float | None = None
    pe_median_trend_1y_vs_3y: float | None = None
    pe_median_trend_1y_vs_5y: float | None = None
    historical_pe_sample_count_1y: int = 0
    historical_pe_sample_count_3y: int = 0
    historical_pe_sample_count_5y: int = 0
    historical_pb_percentile: float | None = None
    pb_percentile_5y: float | None = None
    pe_sample_count: int = 0
    pe_history_years: float | None = None
        
@dataclass
class PriceTargets:

    # =========================
    # Model valuation
    # =========================
    model_buy_price: float | None
    model_fair_price: float | None
    model_sell_price: float | None

    # =========================
    # Historical PE valuation
    # =========================
    
    historical_P05_price: float | None = None
    historical_buy_price: float | None = None
    historical_fair_price: float | None = None
    historical_sell_price: float | None = None
    historical_P95_price: float | None = None
    historical_median_price_1y: float | None = None
    historical_median_price_3y: float | None = None
    historical_median_price_5y: float | None = None

    upside_to_fair: float | None = None

    primary_metric: str = ""
    basis: str = ""
    valuation_environment: str = ""
    historical_reference_period: str | None = None
    historical_reference_percentile: float | None = None
    valuation_applicability: str = ""
    price_recommendation: str = ""
    price_recommendation_confidence: str = ""
    price_recommendation_basis: str = ""
    valuation_eps_base: float | None = None
    ttm_eps_fair_price: float | None = None
    estimated_eps_fair_price: float | None = None
    model_earnings_basis: str = ""

# =========================
# Technical Analysis Models
# =========================

@dataclass
class TechnicalSignal:
    signal: str
    rsi: float | None
    short_ma: float | None
    long_ma: float | None
    support: float | None
    resistance: float | None

# =========================
# Market Flow / Chip Models
# =========================        
        

@dataclass
class ChipSignal:
    signal: str
    stop_loss: float | None
    take_profit: float | None
    reason: str

In [ ]:
# 資料來源：FinMind；Forward EPS、分析師共識、缺漏財務安全性及ROIC例外由 Yahoo 提供
class FinMindTaiwanProvider:
    """台股價格、財報與籌碼：FinMind。Forward EPS、分析師共識、缺漏財務安全性及ROIC例外使用 Yahoo。"""

    BASE_URL = "https://api.finmindtrade.com/api/v4/data"

    def __init__(self, config: dict[str, Any]) -> None:
        settings = config.get("data_sources", {}).get("finmind", {})

        import os
        if settings.get("enabled", True) is False:
            raise ValueError("TW notebook requires FinMind enabled")
        self.token = os.environ.get("FINMIND_TOKEN") or settings.get("token")
        if not self.token:
            raise ValueError("Set FINMIND_TOKEN or data_sources.finmind.token in config/TW.yaml")
        self.config = config
        self._request_cache = {}
        self._issues = {}
        self._forward_eps_cache = {}
        self._yahoo_info_cache = {}
        self._analyst_consensus_cache = {}

        # 台灣上市櫃公司通常面額為 10 元。
        # 如遇特殊面額個股，未來可在 config 加個別 override。
        self.default_par_value = settings.get("par_value_default", 10.0)

        # 個別股票面額 override，例如：
        # ticker_par_values:
        #   "XXXX.TW": 10.0
        self.ticker_par_values = settings.get("ticker_par_values", {})
        
        # 避免同一輪分析重複呼叫 FinMind API
        self._income_cache: dict[str, pd.DataFrame] = {}
        self._balance_cache: dict[str, pd.DataFrame] = {}
        self._cashflow_cache: dict[str, pd.DataFrame] = {}
        self._month_revenue_cache: dict[str, pd.DataFrame] = {}


    @staticmethod
    def ticker_code(ticker: str) -> str:
        import re
        value = str(ticker).strip().upper()
        match = re.fullmatch(r"([0-9]{4,6}[A-Z]?)(?:\.TW|\.TWO)?", value)
        if not match:
            raise ValueError(f"TW notebook requires a Taiwan stock code: {ticker}")
        return match[1]


    def get_financial_statements(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """取得台股損益表資料，並使用記憶體快取。"""

        self.ticker_code(ticker)

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._income_cache:
            self._income_cache[cache_key] = self._request(
                dataset="TaiwanStockFinancialStatements",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._income_cache[cache_key].copy()    
    
    def get_balance_sheet(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """取得台股資產負債表資料，並使用記憶體快取。"""

        self.ticker_code(ticker)

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._balance_cache:
            self._balance_cache[cache_key] = self._request(
                dataset="TaiwanStockBalanceSheet",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._balance_cache[cache_key].copy()
    
    def get_cash_flow_statement(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """
        取得台股現金流量表資料，並使用記憶體快取。

        FinMind Dataset：
        TaiwanStockCashFlowsStatement
        """
        self.ticker_code(ticker)

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._cashflow_cache:
            self._cashflow_cache[cache_key] = self._request(
                dataset="TaiwanStockCashFlowsStatement",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._cashflow_cache[cache_key].copy()
    
    def get_month_revenue(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """
        取得台股月營收資料。

        FinMind Dataset:
        TaiwanStockMonthRevenue

        常見欄位：
        - date
        - stock_id
        - country
        - revenue
        - revenue_month
        - revenue_year
        """
        self.ticker_code(ticker)

        cache_key = f"{ticker}_{start_date}"

        if cache_key not in self._month_revenue_cache:
            self._month_revenue_cache[cache_key] = self._request(
                dataset="TaiwanStockMonthRevenue",
                stock_id=self.ticker_code(ticker),
                start_date=start_date,
            )

        return self._month_revenue_cache[cache_key].copy()
    
    def get_eps_ttm_nowcast_inputs(  
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, float | None]:
        """
        建立 EPS TTM Nowcast 所需資料。

        公式：
        EPS TTM Nowcast
        =
        最近 12 個已公告月營收
        × 最近三季營收加權平均淨利率
        ÷ 最近四季平均股數

        注意：
        - 月營收歸屬月份以 revenue_year / revenue_month 為準。
        - date 通常是公告月份第一天，不能直接視為營收所屬月份。
        """

        result = {
            "ltm_month_revenue": None,
            "weighted_net_margin_3q": None,
            "average_shares_12m": None,
            "eps_ttm_nowcast": None,
        }

        self.ticker_code(ticker)

        # ---------------------------------------------------------
        # 1. 最近 12 個已公告月營收
        # ---------------------------------------------------------
        month_revenue_df = self.get_month_revenue(
            ticker,
            start_date,
        )

        required_month_cols = {
            "revenue",
            "revenue_year",
            "revenue_month",
        }

        if (
            month_revenue_df is None
            or month_revenue_df.empty
            or not required_month_cols.issubset(month_revenue_df.columns)
        ):
            return result

        month_revenue_df = month_revenue_df.copy()

        month_revenue_df["revenue"] = pd.to_numeric(
            month_revenue_df["revenue"],
            errors="coerce",
        )

        month_revenue_df["revenue_year"] = pd.to_numeric(
            month_revenue_df["revenue_year"],
            errors="coerce",
        )

        month_revenue_df["revenue_month"] = pd.to_numeric(
            month_revenue_df["revenue_month"],
            errors="coerce",
        )

        month_revenue_df = month_revenue_df.dropna(
            subset=[
                "revenue",
                "revenue_year",
                "revenue_month",
            ]
        )

        # 用真正的營收歸屬年月建立日期欄位
        month_revenue_df["revenue_period"] = pd.to_datetime(
            {
                "year": month_revenue_df["revenue_year"].astype(int),
                "month": month_revenue_df["revenue_month"].astype(int),
                "day": 1,
            }
        )

        month_revenue_df = (
            month_revenue_df
            .sort_values("revenue_period")
            .drop_duplicates(
                subset=["revenue_period"],
                keep="last",
            )
            .reset_index(drop=True)
        )

        latest_12_months = month_revenue_df.tail(12)

        # 必須有完整 12 個月才建立 Nowcast
        if len(latest_12_months) != 12:
            LOGGER.warning(
                "FinMind month revenue has fewer than 12 observations for %s",
                ticker,
            )
            return result

        ltm_month_revenue = float(
            latest_12_months["revenue"].sum()
        )

        result["ltm_month_revenue"] = ltm_month_revenue

        # ---------------------------------------------------------
        # 2. 最近三季營收加權平均淨利率
        # ---------------------------------------------------------
        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        required_income_cols = {
            "date",
            "type",
            "value",
        }

        if (
            income_df is None
            or income_df.empty
            or not required_income_cols.issubset(income_df.columns)
        ):
            return result

        income_df = income_df.copy()

        income_df["date"] = pd.to_datetime(
            income_df["date"],
            errors="coerce",
        )

        income_df["value"] = pd.to_numeric(
            income_df["value"],
            errors="coerce",
        )

        income_df = income_df.dropna(
            subset=["date", "value"]
        )

        report_dates = sorted(
            pd.Timestamp(item)
            for item in income_df["date"].dropna().unique()
        )

        # 最近 3 個已公告季度
        latest_3_quarters = report_dates[-3:]

        if len(latest_3_quarters) != 3:
            return result

        revenue_values = [
            self._pick_type_value(
                income_df,
                report_date,
                ["Revenue"],
            )
            for report_date in latest_3_quarters
        ]

        net_income_values = [
            self._pick_type_value(
                income_df,
                report_date,
                [
                    "NetIncome",
                    "IncomeAfterTaxes",
                ],
            )
            for report_date in latest_3_quarters
        ]

        if (
            any(value is None for value in revenue_values)
            or any(value is None for value in net_income_values)
        ):
            return result

        revenue_3q = float(sum(revenue_values))
        net_income_3q = float(sum(net_income_values))

        weighted_net_margin_3q = safe_divide(
            net_income_3q,
            revenue_3q,
        )

        if weighted_net_margin_3q is None:
            return result

        # 防止資料異常，正常非金融公司可容許到 60%
        weighted_net_margin_3q = max(
            0.0,
            min(0.60, weighted_net_margin_3q),
        )

        result["weighted_net_margin_3q"] = (
            weighted_net_margin_3q
        )

        # ---------------------------------------------------------
        # 3. 最近四季平均股數
        # ---------------------------------------------------------
        fundamentals_df = self.get_historical_fundamentals(
            ticker,
            start_date,
        )

        if (
            fundamentals_df is None
            or fundamentals_df.empty
            or "shares_outstanding" not in fundamentals_df.columns
        ):
            return result

        valid_shares = (
            fundamentals_df
            .dropna(subset=["shares_outstanding"])
            .sort_values("report_date")
        )

        valid_shares = valid_shares[
            pd.to_numeric(
                valid_shares["shares_outstanding"],
                errors="coerce",
            ) > 0
        ]

        recent_shares = valid_shares.tail(4)[
            "shares_outstanding"
        ]

        if len(recent_shares) < 2:
            return result

        average_shares_12m = float(
            pd.to_numeric(
                recent_shares,
                errors="coerce",
            ).mean()
        )

        if average_shares_12m <= 0:
            return result

        result["average_shares_12m"] = average_shares_12m

        # ---------------------------------------------------------
        # 4. 最終 EPS TTM Nowcast
        # ---------------------------------------------------------
        estimated_net_income = (
            ltm_month_revenue
            * weighted_net_margin_3q
        )

        eps_ttm_nowcast = safe_divide(
            estimated_net_income,
            average_shares_12m,
        )

        result["eps_ttm_nowcast"] = eps_ttm_nowcast

        return result
    
    
    
    def get_cashflow_history(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, list[float]]:
        """
        建立台股現金流歷史資料。

        FinMind 現金流量表中的：
        - CashFlowsFromOperatingActivities：年初至今累積營業現金流
        - PropertyAndPlantAndEquipment：年初至今累積資本支出，通常為負數

        本函式會轉成單季資料，並回傳最新到最舊的序列。

        回傳：
        {
            "quarterly_operating_cash_flow": [...],
            "quarterly_capex": [...],            # 正數，代表支出
            "quarterly_free_cash_flow": [...],
            "operating_cash_flow": [...],        # 完整年度 OCF
            "capex": [...],                      # 完整年度 CapEx，正數
            "free_cash_flow": [...],             # 完整年度 FCF
        }
        """

        empty_history = {
            "quarterly_operating_cash_flow": [],
            "quarterly_capex": [],
            "quarterly_free_cash_flow": [],
            "operating_cash_flow": [],
            "capex": [],
            "free_cash_flow": [],
        }

        self.ticker_code(ticker)

        cashflow_df = self.get_cash_flow_statement(
            ticker,
            start_date,
        )

        required_cols = {"date", "type", "value"}

        if (
            cashflow_df is None
            or cashflow_df.empty
            or not required_cols.issubset(cashflow_df.columns)
        ):
            LOGGER.warning(
                "FinMind cashflow history unavailable for %s",
                ticker,
            )
            return empty_history

        cashflow_df = cashflow_df.copy()

        cashflow_df["date"] = pd.to_datetime(
            cashflow_df["date"],
            errors="coerce",
        )

        cashflow_df["value"] = pd.to_numeric(
            cashflow_df["value"],
            errors="coerce",
        )

        cashflow_df = cashflow_df.dropna(
            subset=["date", "value"]
        )

        if cashflow_df.empty:
            return empty_history

        # FinMind 已確認的主要欄位。
        # 若未來某公司有不同名稱，可於此補候選名稱。
        ocf_candidates = [
            "CashFlowsFromOperatingActivities",
            "NetCashInflowFromOperatingActivities",
        ]

        capex_candidates = [
            "PropertyAndPlantAndEquipment",
        ]

        report_dates = sorted(
            pd.Timestamp(item)
            for item in cashflow_df["date"].dropna().unique()
        )

        cumulative_ocf = [
            self._pick_type_value(
                cashflow_df,
                report_date,
                ocf_candidates,
            )
            for report_date in report_dates
        ]

        cumulative_capex_raw = [
            self._pick_type_value(
                cashflow_df,
                report_date,
                capex_candidates,
            )
            for report_date in report_dates
        ]

        # 現金流為累積值，轉為單季值。
        quarterly_ocf = self._quarterly_from_cumulative(
            cumulative_ocf,
            report_dates,
        )

        quarterly_capex_signed = self._quarterly_from_cumulative(
            cumulative_capex_raw,
            report_dates,
        )

        # PropertyAndPlantAndEquipment 通常為負數，轉成正的 CapEx 支出。
        quarterly_capex = [
            abs(value) if value is not None else None
            for value in quarterly_capex_signed
        ]

        quarterly_fcf = []

        for ocf, capex in zip(
            quarterly_ocf,
            quarterly_capex,
        ):
            if ocf is None or capex is None:
                quarterly_fcf.append(None)
            else:
                quarterly_fcf.append(float(ocf - capex))

        # ---------------------------------------------------------
        # 建立完整年度 OCF / CapEx / FCF
        # 只採用有完整 Q1~Q4 的年度，避免當年度尚未結束而誤算
        # ---------------------------------------------------------
        quarter_df = pd.DataFrame(
            {
                "date": report_dates,
                "quarterly_ocf": quarterly_ocf,
                "quarterly_capex": quarterly_capex,
                "quarterly_fcf": quarterly_fcf,
            }
        )

        quarter_df["year"] = quarter_df["date"].dt.year
        quarter_df["quarter"] = quarter_df["date"].dt.quarter

        annual_rows: list[dict[str, Any]] = []

        for year, group in quarter_df.groupby("year"):
            group = group.sort_values("quarter")

            available_quarters = set(
                group["quarter"]
                .dropna()
                .astype(int)
                .tolist()
            )

            # 只有完整年度才納入 CAGR 歷史
            if available_quarters != {1, 2, 3, 4}:
                continue

            ocf_values = group["quarterly_ocf"].dropna()
            capex_values = group["quarterly_capex"].dropna()
            fcf_values = group["quarterly_fcf"].dropna()

            annual_rows.append(
                {
                    "year": int(year),
                    "operating_cash_flow": (
                        float(ocf_values.sum())
                        if len(ocf_values) == 4
                        else None
                    ),
                    "capex": (
                        float(capex_values.sum())
                        if len(capex_values) == 4
                        else None
                    ),
                    "free_cash_flow": (
                        float(fcf_values.sum())
                        if len(fcf_values) == 4
                        else None
                    ),
                }
            )

        annual_df = pd.DataFrame(annual_rows)

        if annual_df.empty:
            annual_ocf = []
            annual_capex = []
            annual_fcf = []
        else:
            annual_df = annual_df.sort_values(
                "year",
                ascending=False,
            )

            annual_ocf = (
                annual_df["operating_cash_flow"]
                .dropna()
                .astype(float)
                .tolist()
            )

            annual_capex = (
                annual_df["capex"]
                .dropna()
                .astype(float)
                .tolist()
            )

            annual_fcf = (
                annual_df["free_cash_flow"]
                .dropna()
                .astype(float)
                .tolist()
            )

        # QualityEngine 的歷史資料假設：最新 → 最舊
        return {
            "quarterly_operating_cash_flow": [
                float(value) if value is not None else None
                for value in reversed(quarterly_ocf)
            ],
            "quarterly_capex": [
                float(value) if value is not None else None
                for value in reversed(quarterly_capex)
            ],
            "quarterly_free_cash_flow": [
                float(value) if value is not None else None
                for value in reversed(quarterly_fcf)
            ],
            "operating_cash_flow": annual_ocf,
            "capex": annual_capex,
            "free_cash_flow": annual_fcf,
        }

    def get_snapshot_financial_fields(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, float | None]:
        """
        建立可覆蓋 FinancialSnapshot 的最新財務欄位。

        損益表類欄位採最近四季合計 TTM：
        - revenue
        - gross_profit
        - operating_income
        - net_income
        - interest_expense（若可取得）

        資產負債表類欄位採最新可得季度：
        - total_equity
        - cash
        - total_debt
        - current_assets
        - current_liabilities
        - inventory

        注意：
        若某欄位資料不足，回傳 None，不切換其他來源。
        """

        result = {
            "revenue": None,
            "gross_profit": None,
            "operating_income": None,
            "net_income": None,
            "interest_expense": None,
            "total_equity": None,
            "cash": None,
            "total_debt": None,
            "current_assets": None,
            "current_liabilities": None,
            "inventory": None,
        }

        self.ticker_code(ticker)

        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        balance_df = self.get_balance_sheet(
            ticker,
            start_date,
        )

        required_cols = {"date", "type", "value"}

        # ---------------------------------------------------------
        # 損益表：最近四季 TTM
        # ---------------------------------------------------------
        if (
            income_df is not None
            and not income_df.empty
            and required_cols.issubset(income_df.columns)
        ):
            income_df = income_df.copy()

            income_df["date"] = pd.to_datetime(
                income_df["date"],
                errors="coerce",
            )

            income_df["value"] = pd.to_numeric(
                income_df["value"],
                errors="coerce",
            )

            income_df = income_df.dropna(
                subset=["date", "value"]
            )

            if not income_df.empty:
                # 使用已驗證為單季值的損益表項目
                metric_candidates = {
                    "revenue": [
                        "Revenue",
                    ],
                    "gross_profit": [
                        "GrossProfit",
                    ],
                    "operating_income": [
                        "OperatingIncome",
                    ],
                    "net_income": [
                        "NetIncome",
                        "IncomeAfterTaxes",
                    ],
                    # FinMind 不同公司可能未提供此欄；缺值即回傳 None
                    "interest_expense": [
                        "InterestExpense",
                        "FinanceCosts",
                        "InterestAndFinanceCosts",
                    ],
                }

                report_dates = sorted(
                    pd.Timestamp(item)
                    for item in income_df["date"].dropna().unique()
                )

                # 最近四個財報季度
                latest_four_dates = report_dates[-4:]

                if len(latest_four_dates) == 4:
                    for field_name, candidates in metric_candidates.items():
                        values = [
                            self._pick_type_value(
                                income_df,
                                report_date,
                                candidates,
                            )
                            for report_date in latest_four_dates
                        ]

                        # TTM 必須四季都有資料，避免用部分年度資料誤算
                        if all(value is not None for value in values):
                            result[field_name] = float(sum(values))

        # ---------------------------------------------------------
        # 資產負債表：最新可得季度
        # ---------------------------------------------------------
        if (
            balance_df is not None
            and not balance_df.empty
            and required_cols.issubset(balance_df.columns)
        ):
            balance_df = balance_df.copy()

            balance_df["date"] = pd.to_datetime(
                balance_df["date"],
                errors="coerce",
            )

            balance_df["value"] = pd.to_numeric(
                balance_df["value"],
                errors="coerce",
            )

            balance_df = balance_df.dropna(
                subset=["date", "value"]
            )

            if not balance_df.empty:
                latest_balance_date = balance_df["date"].max()

                balance_candidates = {
                    "total_equity": [
                        "EquityAttributableToOwnersOfParent",
                        "Equity",
                    ],
                    "cash": [
                        "CashAndCashEquivalents",
                    ],
                    "current_assets": [
                        "CurrentAssets",
                    ],
                    "current_liabilities": [
                        "CurrentLiabilities",
                    ],
                    "inventory": [
                        "Inventories",
                        "Inventory",
                    ],
                }

                for field_name, candidates in balance_candidates.items():
                    result[field_name] = self._pick_type_value(
                        balance_df,
                        latest_balance_date,
                        candidates,
                    )

                # 台股財報中的 Total Debt 不一定有統一欄位，
                # 先以短債 + 長債 + 公司債作簡化估計。
                short_debt = self._pick_type_value(
                    balance_df,
                    latest_balance_date,
                    [
                        "ShorttermBorrowings",
                        "ShortTermBorrowings",
                    ],
                )

                long_debt = self._pick_type_value(
                    balance_df,
                    latest_balance_date,
                    [
                        "LongtermBorrowings",
                        "LongTermBorrowings",
                    ],
                )

                bonds_payable = self._pick_type_value(
                    balance_df,
                    latest_balance_date,
                    [
                        "BondsPayable",
                    ],
                )

                debt_values = [
                    value
                    for value in [
                        short_debt,
                        long_debt,
                        bonds_payable,
                    ]
                    if value is not None
                ]

                if len(debt_values) == 3:
                    result["total_debt"] = float(sum(debt_values))

        return result
    
    @staticmethod
    def _pick_type_value(
        df: pd.DataFrame,
        report_date: pd.Timestamp,
        candidates: list[str],
    ) -> float | None:
        """
        從 FinMind 長格式資料中，
        尋找指定日期與候選 type 名稱的第一個有效數值。
        """

        if df is None or df.empty:
            return None

        required_cols = {"date", "type", "value"}
        if not required_cols.issubset(df.columns):
            return None

        date_value = pd.Timestamp(report_date).normalize()

        subset = df[
            df["date"].dt.normalize() == date_value
        ]

        for item_type in candidates:
            match = subset[
                subset["type"] == item_type
            ]

            if match.empty:
                continue

            values = pd.to_numeric(
                match["value"],
                errors="coerce",
            ).dropna()

            if not values.empty:
                return float(values.iloc[0])

        return None

    @staticmethod
    def _quarterly_from_cumulative(
        cumulative_values: list[float | None],
        dates: list[pd.Timestamp],
    ) -> list[float | None]:
        """
        把台灣常見的「年初至今累積 EPS」轉成單季 EPS。

        輸入必須按時間由舊到新排序。

        規則：
        - 新年度第一筆：直接視為該年度第一個季度的 EPS
        - 同年度後續資料：本期累積 EPS - 前期累積 EPS
        """

        quarterly_values: list[float | None] = []

        previous_cumulative: float | None = None
        previous_year: int | None = None

        for report_date, cumulative in zip(dates, cumulative_values):
            current_year = pd.Timestamp(report_date).year

            if cumulative is None:
                quarterly_values.append(None)
                previous_cumulative = None
                previous_year = current_year
                continue

            # 新年度第一筆，或上一筆資料無效
            if previous_year != current_year or previous_cumulative is None:
                quarterly_values.append(float(cumulative))
            else:
                quarterly_values.append(
                    float(cumulative - previous_cumulative)
                )

            previous_cumulative = float(cumulative)
            previous_year = current_year

        return quarterly_values

    def _shares_from_capital(
        self,
        ticker: str,
        balance_df: pd.DataFrame,
        report_date: pd.Timestamp,
    ) -> float | None:
        """
        以股本 / 面額推估期末發行股數。

        FinMind 的 CapitalStock / OrdinaryShare 通常為股本金額，
        非直接股數，因此以：
            shares = capital_stock / par_value
        推估。

        台灣大多數公司面額為 10 元。
        """

        capital_candidates = [
            "CapitalStock",
            "OrdinaryShare",
        ]

        capital_stock = self._pick_type_value(
            balance_df,
            report_date,
            capital_candidates,
        )

        par_value = self.ticker_par_values.get(
            ticker,
            self.default_par_value,
        )

        if capital_stock is None or par_value in (None, 0):
            return None

        return safe_divide(capital_stock, par_value)

    def get_historical_fundamentals(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> pd.DataFrame:
        """
        回傳標準化歷史基本面資料。

        回傳欄位：
        - report_date
        - ttm_eps
        - bvps

        額外保留欄位供 debug：
        - quarterly_eps
        - cumulative_eps
        - total_equity
        - shares_outstanding
        """
        
        columns = [
            "report_date",
            "ttm_eps",
            "bvps",
            "quarterly_eps",
            "reported_eps",
            "total_equity",
            "shares_outstanding",
        ]

        empty = pd.DataFrame(columns=columns)

        self.ticker_code(ticker)

        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        balance_df = self.get_balance_sheet(
            ticker,
            start_date,
        )

        if income_df.empty:
            LOGGER.warning(
                "FinMind income statement is empty for %s",
                ticker,
            )
            return empty

        required_cols = {"date", "type", "value"}
        if not required_cols.issubset(income_df.columns):
            LOGGER.warning(
                "FinMind income data format invalid for %s",
                ticker,
            )
            return empty

        income_df = income_df.copy()
        income_df["date"] = pd.to_datetime(
            income_df["date"],
            errors="coerce",
        )
        income_df["value"] = pd.to_numeric(
            income_df["value"],
            errors="coerce",
        )
        income_df = income_df.dropna(
            subset=["date"]
        )

        if not balance_df.empty:
            if required_cols.issubset(balance_df.columns):
                balance_df = balance_df.copy()
                balance_df["date"] = pd.to_datetime(
                    balance_df["date"],
                    errors="coerce",
                )
                balance_df["value"] = pd.to_numeric(
                    balance_df["value"],
                    errors="coerce",
                )
                balance_df = balance_df.dropna(
                    subset=["date"]
                )
            else:
                LOGGER.warning(
                    "FinMind balance sheet format invalid for %s",
                    ticker,
                )
                balance_df = pd.DataFrame()

        report_dates = sorted(
            pd.Timestamp(item)
            for item in income_df["date"].dropna().unique()
        )

        # 依剛剛測試結果，FinMind 已確認 type 為 EPS。
        eps_candidates = [
            "EPS",
        ]

        # 優先使用歸屬母公司業主權益，較適合每股淨值。
        equity_candidates = [
            "EquityAttributableToOwnersOfParent",
            "Equity",
        ]
        
        # FinMind TaiwanStockFinancialStatements 的 EPS 經驗證後為「單季 EPS」。
        # 不要再當作累積 EPS 做差分。
        reported_eps = [
            self._pick_type_value(
                income_df,
                report_date,
                eps_candidates,
            )
            for report_date in report_dates
        ]

        quarterly_eps = reported_eps.copy()
        
        rows: list[dict[str, Any]] = []

        for index, report_date in enumerate(report_dates):
            # TTM EPS = 最近四季單季 EPS 加總
            ttm_eps = None

            if index >= 3:
                eps_window = quarterly_eps[
                    index - 3:index + 1
                ]

                if all(value is not None for value in eps_window):
                    ttm_eps = float(sum(eps_window))

            total_equity = None
            shares_outstanding = None
            bvps = None

            if not balance_df.empty:
                total_equity = self._pick_type_value(
                    balance_df,
                    report_date,
                    equity_candidates,
                )

                shares_outstanding = self._shares_from_capital(
                    ticker,
                    balance_df,
                    report_date,
                )

                bvps = safe_divide(
                    total_equity,
                    shares_outstanding,
                )
                        
            rows.append(
                {
                    "report_date": report_date,
                    "ttm_eps": ttm_eps,
                    "bvps": bvps,
                    "quarterly_eps": quarterly_eps[index],
                    "reported_eps": reported_eps[index],
                    "total_equity": total_equity,
                    "shares_outstanding": shares_outstanding,
                }
            )

        result = pd.DataFrame(rows)

        if result.empty:
            return empty

        result = (
            result
            .sort_values("report_date")
            .reset_index(drop=True)
        )

        valid_ttm_eps = result["ttm_eps"].notna().sum()
        valid_bvps = result["bvps"].notna().sum()

        LOGGER.info(
            "FinMind fundamentals built for %s: TTM EPS=%s, BVPS=%s",
            ticker,
            valid_ttm_eps,
            valid_bvps,
        )

        return result
    

    def get_growth_history(
        self,
        ticker: str,
        start_date: str = "2018-01-01",
    ) -> dict[str, list[float]]:
        """
        建立給 QualityEngine 使用的成長資料。

        回傳格式與既有 snapshot.financial_history 相容：

        {
            "quarterly_revenue": [...],  # 最新到最舊，單季營收
            "quarterly_eps": [...],      # 最新到最舊，單季 EPS
            "revenue": [...],            # 最新到最舊，完整年度營收
            "eps": [...],                # 最新到最舊，完整年度 EPS
        }

        注意：
        - FinMind 的 EPS 已驗證為單季 EPS
        - Revenue 需先用 Step 0 驗證為單季營收
        - 年度資料只採用完整四季，避免 2026 Q1 被誤當成年營收
        """

        empty_history = {
            "quarterly_revenue": [],
            "quarterly_eps": [],
            "revenue": [],
            "eps": [],
        }

        self.ticker_code(ticker)

        income_df = self.get_financial_statements(
            ticker,
            start_date,
        )

        required_cols = {"date", "type", "value"}

        if (
            income_df is None
            or income_df.empty
            or not required_cols.issubset(income_df.columns)
        ):
            LOGGER.warning(
                "FinMind growth history unavailable for %s",
                ticker,
            )
            return empty_history

        income_df = income_df.copy()

        income_df["date"] = pd.to_datetime(
            income_df["date"],
            errors="coerce",
        )

        income_df["value"] = pd.to_numeric(
            income_df["value"],
            errors="coerce",
        )

        income_df = income_df.dropna(
            subset=["date", "value"]
        )

        if income_df.empty:
            return empty_history

        # 將長格式轉成每季一列的寬格式
        quarterly = (
            income_df[
                income_df["type"].isin(["Revenue", "EPS"])
            ]
            .pivot_table(
                index="date",
                columns="type",
                values="value",
                aggfunc="first",
            )
            .reset_index()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if quarterly.empty:
            return empty_history

        # 確保欄位存在，避免某些公司缺 Revenue 或 EPS 時報錯
        if "Revenue" not in quarterly.columns:
            quarterly["Revenue"] = pd.NA

        if "EPS" not in quarterly.columns:
            quarterly["EPS"] = pd.NA

        quarterly["Revenue"] = pd.to_numeric(
            quarterly["Revenue"],
            errors="coerce",
        )

        quarterly["EPS"] = pd.to_numeric(
            quarterly["EPS"],
            errors="coerce",
        )

        # ------------------------------------------------------
        # 季資料：最新 → 最舊
        # QualityEngine._period_growth() 假設 values[0] 為最新一期
        # ------------------------------------------------------
        quarterly_desc = quarterly.sort_values(
            "date",
            ascending=False,
        ).reset_index(drop=True)

        quarterly_revenue = (
            quarterly_desc["Revenue"]
            .dropna()
            .astype(float)
            .tolist()
        )

        quarterly_eps = (
            quarterly_desc["EPS"]
            .dropna()
            .astype(float)
            .tolist()
        )

        # ------------------------------------------------------
        # 年資料：只有完整 4 季才納入
        # 防止目前年度例如 2026 年只有 Q1，被誤判為完整年度資料
        # ------------------------------------------------------
        quarterly["year"] = quarterly["date"].dt.year
        quarterly["quarter"] = quarterly["date"].dt.quarter

        annual_rows: list[dict[str, Any]] = []

        for year, group in quarterly.groupby("year"):
            group = group.sort_values("quarter")

            available_quarters = set(
                group["quarter"].dropna().astype(int).tolist()
            )

            # 只採完整 Q1/Q2/Q3/Q4 年度
            if available_quarters != {1, 2, 3, 4}:
                continue

            revenue_values = group["Revenue"].dropna()
            eps_values = group["EPS"].dropna()

            annual_revenue = (
                float(revenue_values.sum())
                if len(revenue_values) == 4
                else None
            )

            annual_eps = (
                float(eps_values.sum())
                if len(eps_values) == 4
                else None
            )

            annual_rows.append(
                {
                    "year": int(year),
                    "revenue": annual_revenue,
                    "eps": annual_eps,
                }
            )

        annual_df = pd.DataFrame(annual_rows)

        if annual_df.empty:
            annual_revenue = []
            annual_eps = []
        else:
            annual_df = annual_df.sort_values(
                "year",
                ascending=False,
            )

            annual_revenue = (
                annual_df["revenue"]
                .dropna()
                .astype(float)
                .tolist()
            )

            annual_eps = (
                annual_df["eps"]
                .dropna()
                .astype(float)
                .tolist()
            )

        return {
            "quarterly_revenue": quarterly_revenue,
            "quarterly_eps": quarterly_eps,
            "revenue": annual_revenue,
            "eps": annual_eps,
        }

    source_name = "FinMind"

    def _issue(self, stock_id: str, message: str) -> None:
        messages = self._issues.setdefault(stock_id, [])
        if message not in messages:
            messages.append(message)
            LOGGER.warning("FinMind %s: %s", stock_id, message)

    def _request(self, dataset: str, stock_id: str = "", start_date: str = "2018-01-01") -> pd.DataFrame:
        """Only FinMind API; failed requests never switch sources or expose credentials."""
        import requests
        key = (dataset, stock_id, start_date)
        if key in self._request_cache:
            return self._request_cache[key].copy()
        params = {"dataset": dataset}
        if stock_id:
            params["data_id"] = stock_id
            params["start_date"] = start_date
        try:
            response = requests.get(
                self.BASE_URL, params=params,
                headers={"Authorization": f"Bearer {self.token}"}, timeout=30,
            )
            if response.status_code != 200:
                self._issue(stock_id, f"{dataset}: HTTP {response.status_code}")
                return pd.DataFrame()
            payload = response.json()
            if payload.get("status", 200) != 200:
                self._issue(stock_id, f"{dataset}: API status {payload.get('status')}")
                return pd.DataFrame()
            rows = payload.get("data")
            if not isinstance(rows, list) or not rows:
                self._issue(stock_id, f"{dataset}: no data")
                return pd.DataFrame()
            frame = pd.DataFrame(rows)
            self._request_cache[key] = frame
            return frame.copy()
        except (requests.RequestException, ValueError) as exc:
            self._issue(stock_id, f"{dataset}: {type(exc).__name__}")
            return pd.DataFrame()

    def get_price_history(self, ticker: str, period: str = "5y") -> pd.DataFrame:
        """Unadjusted daily OHLCV, matching the original auto_adjust=False basis."""
        import re
        code = self.ticker_code(ticker)
        match = re.fullmatch(r"([1-9][0-9]*)(y|mo|d)", period)
        if not match:
            raise ValueError("period must be a positive number followed by y, mo or d")
        amount, unit = int(match[1]), match[2]
        offset = pd.DateOffset(**{{"y": "years", "mo": "months", "d": "days"}[unit]: amount})
        start = (pd.Timestamp.today().normalize() - offset).date().isoformat()
        raw = self._request("TaiwanStockPrice", code, start)
        columns = {"open": "Open", "max": "High", "min": "Low", "close": "Close", "Trading_Volume": "Volume"}
        empty = pd.DataFrame(columns=list(columns.values()), index=pd.DatetimeIndex([], name="Date"))
        if raw.empty:
            return empty
        if not {"date", *columns}.issubset(raw.columns):
            self._issue(code, "TaiwanStockPrice: missing OHLCV columns")
            return empty
        frame = raw.rename(columns=columns).copy()
        frame["Date"] = pd.to_datetime(frame["date"], errors="coerce")
        for column in columns.values():
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
            frame.loc[~frame[column].map(lambda x: pd.notna(x) and math.isfinite(x)), column] = float("nan")
        frame = frame.dropna(subset=["Date"]).drop_duplicates("Date", keep="last").set_index("Date").sort_index()
        # No announced price is encoded as zero by FinMind; do not treat it as a crash.
        frame = frame[(frame["Close"] > 0) & (frame.index >= pd.Timestamp(start))]
        frame.loc[frame["Volume"] < 0, "Volume"] = float("nan")
        for column in ("Open", "High", "Low"):
            frame.loc[frame[column] <= 0, column] = float("nan")
        return frame[list(columns.values())]

    def get_chip_data(self, ticker: str) -> pd.DataFrame:
        """Daily institutional net shares. Missing categories/days remain missing."""
        code = self.ticker_code(ticker)
        prices = self.get_price_history(ticker)
        columns = ["date", "institutional_net_buy"]
        if prices.empty:
            return pd.DataFrame(columns=columns)
        calendar = prices.index[-max(30, self.config.get("chip_analysis", {}).get("institutional_buying_days", 3)):]
        raw = self._request("TaiwanStockInstitutionalInvestorsBuySell", code, calendar.min().date().isoformat())
        result = pd.DataFrame({"date": calendar, "institutional_net_buy": float("nan")})
        if raw.empty or not {"date", "name", "buy", "sell"}.issubset(raw.columns):
            self._issue(code, "Institutional flow unavailable")
            return result
        raw = raw.copy()
        raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
        raw["net"] = pd.to_numeric(raw["buy"], errors="coerce") - pd.to_numeric(raw["sell"], errors="coerce")
        raw = raw.dropna(subset=["date"]).drop_duplicates(["date", "name"], keep="last")
        wide = raw.pivot(index="date", columns="name", values="net")
        # Recent listed/OTC data: all five categories must be present.
        required = ["Foreign_Investor", "Foreign_Dealer_Self", "Investment_Trust", "Dealer_self", "Dealer_Hedging"]
        net = wide.reindex(columns=required).sum(axis=1, min_count=len(required))
        result["institutional_net_buy"] = result["date"].map(net)
        result.loc[~result["institutional_net_buy"].map(lambda x: pd.notna(x) and math.isfinite(x)), "institutional_net_buy"] = float("nan")
        if result["institutional_net_buy"].tail(self.config.get("chip_analysis", {}).get("institutional_buying_days", 3)).isna().any():
            self._issue(code, "Recent institutional flow incomplete or not yet published")
        return result

    def get_margin_data(self, ticker: str, start_date: str = "2018-01-01") -> pd.DataFrame:
        """FinMind raw margin data; not included in the existing scoring formula."""
        return self._request("TaiwanStockMarginPurchaseShortSale", self.ticker_code(ticker), start_date)

    @staticmethod
    def _latest_number(frame: pd.DataFrame, column: str) -> float | None:
        if frame.empty or column not in frame:
            return None
        value = pd.to_numeric(pd.Series([frame.iloc[-1][column]]), errors="coerce").iloc[0]
        return float(value) if pd.notna(value) and math.isfinite(value) else None

    def get_snapshot(self, ticker: str) -> FinancialSnapshot:
        code = self.ticker_code(ticker)
        prices = self.get_price_history(ticker)
        if prices.empty:
            raise ValueError(f"FinMind price unavailable for {ticker}; no alternate source is enabled")
        snapshot = FinancialSnapshot(ticker=ticker, source=self.source_name, as_of=prices.index[-1].date(), current_price=float(prices["Close"].iloc[-1]))
        recent = prices.loc[prices.index >= prices.index[-1] - pd.Timedelta(weeks=52)]
        high, low = recent["High"].max(), recent["Low"].min()
        snapshot.week_52_high = float(high) if pd.notna(high) else None
        snapshot.week_52_low = float(low) if pd.notna(low) else None
        if (pd.Timestamp.today().normalize() - prices.index[-1]).days > 7:
            self._issue(code, "Latest daily close is more than 7 calendar days old")
        info = self._request("TaiwanStockInfo")
        if not info.empty and {"stock_id", "industry_category"}.issubset(info.columns):
            selected = info[info["stock_id"].astype(str) == code]
            if "date" in selected:
                selected = selected.sort_values("date")
            if not selected.empty:
                snapshot.asset_type = "ETF" if selected.iloc[-1]["industry_category"] == "ETF" else "EQUITY"
        fundamentals = self.get_historical_fundamentals(ticker).sort_values("report_date")
        snapshot.ttm_eps = snapshot.eps = self._latest_number(fundamentals, "ttm_eps")
        snapshot.book_value = snapshot.book_value_per_share = self._latest_number(fundamentals, "bvps")
        snapshot.shares_outstanding = self._latest_number(fundamentals, "shares_outstanding")
        if snapshot.shares_outstanding is not None and snapshot.shares_outstanding <= 0:
            snapshot.shares_outstanding = None
        if "shares_outstanding" in fundamentals:
            shares = pd.to_numeric(fundamentals["shares_outstanding"].tail(4), errors="coerce")
            if len(shares) == 4 and shares.notna().all() and (shares > 0).all():
                snapshot.average_shares_12m = float(shares.mean())
        snapshot.financial_history.update(self.get_growth_history(ticker))
        cashflow = self.get_cashflow_history(ticker)
        snapshot.financial_history.update(cashflow)
        for field_name in ("operating_cash_flow", "capex", "free_cash_flow"):
            values = cashflow.get(f"quarterly_{field_name}", [])[:4]
            if len(values) == 4 and all(v is not None and math.isfinite(v) for v in values):
                setattr(snapshot, field_name, float(sum(values)))
        for field_name, value in self.get_snapshot_financial_fields(ticker).items():
            if value is not None and math.isfinite(value):
                setattr(snapshot, field_name, float(value))
        if snapshot.shares_outstanding is not None:
            snapshot.market_cap = snapshot.current_price * snapshot.shares_outstanding
        # Estimated EV excludes minority interest/preferred stock; missing inputs stay missing.
        if all(v is not None for v in (snapshot.market_cap, snapshot.total_debt, snapshot.cash)):
            snapshot.enterprise_value = snapshot.market_cap + snapshot.total_debt - snapshot.cash
        settings = self.config.get("eps_ttm_nowcast", {})
        sector = self.config.get("ticker_classification", {}).get(ticker, {}).get("sector")
        if settings.get("enabled", True) and sector not in settings.get("disabled_sectors", []):
            nowcast = self.get_eps_ttm_nowcast_inputs(ticker)
            value = nowcast.get("eps_ttm_nowcast")
            if value is not None and math.isfinite(value):
                snapshot.eps_ttm_nowcast = float(value)
                if value > 0:
                    snapshot.pe_ttm_nowcast = safe_divide(snapshot.current_price, value)
        snapshot.quarterly_reference = self.get_quarterly_reference(ticker)
        try:
            snapshot.cashflow_quality, snapshot.fundamental_review_inputs = self.get_fundamental_review_bundle(ticker, snapshot, info)
        except Exception as exc:
            snapshot.cashflow_quality = {"notes": [f"附加現金品質資料失敗：{type(exc).__name__}"]}
            snapshot.fundamental_review_inputs = {"note": f"附加評價資料失敗：{type(exc).__name__}"}
        snapshot.forward_eps_meta = self.get_forward_eps(ticker)
        snapshot.forward_eps = snapshot.forward_eps_meta["value"]
        if snapshot.forward_eps is not None:
            snapshot.source = "FinMind + Yahoo Forward EPS only"
        else:
            self._issue(code, snapshot.forward_eps_meta["status"])
        if snapshot.fundamental_review_inputs.get("ebitda_source") == "Yahoo年度EBITDA":
            snapshot.source = "FinMind + Yahoo Forward EPS / EBITDA" if snapshot.forward_eps is not None else "FinMind + Yahoo EBITDA"
        self.supplement_safety_and_roic(ticker, snapshot)
        if "Yahoo financial safety" in snapshot.source or "Yahoo ROIC" in snapshot.source:
            snapshot.source = snapshot.source.replace(" only", "")
        missing = [name for name in ("ttm_eps", "book_value_per_share", "revenue", "net_income", "free_cash_flow", "shares_outstanding", "total_debt") if getattr(snapshot, name) is None]
        if missing:
            self._issue(code, "Missing fields: " + ", ".join(missing))
        return snapshot

    def get_data_notes(self, ticker: str) -> str:
        return "; ".join(self._issues.get("", []) + self._issues.get(self.ticker_code(ticker), []))

    def get_quarterly_reference(self, ticker: str) -> dict[str, Any]:
        """Calendar-quarter observations, newest first; gaps never shift quarter positions.

        FinMind income EPS/revenue are single-quarter values in the existing model.
        OCF is year-to-date: Q1 is direct, Q2-Q4 subtract the preceding YTD quarter.
        These reference observations are separate from annual/TTM scoring inputs.
        """
        self.ticker_code(ticker)
        income = self.get_financial_statements(ticker)
        cashflow = self.get_cash_flow_statement(ticker)

        def quarterly_table(raw):
            if raw is None or raw.empty or not {"date", "type", "value"}.issubset(raw.columns):
                return pd.DataFrame()
            frame = raw.copy()
            frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
            frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
            frame = frame.dropna(subset=["date", "type"]).sort_values("date", kind="stable")
            frame["quarter"] = frame["date"].dt.to_period("Q-DEC")
            frame = frame.drop_duplicates(["quarter", "type"], keep="last")
            return frame.pivot(index="quarter", columns="type", values="value").sort_index()

        inc = quarterly_table(income)
        cash = quarterly_table(cashflow)
        available = list(inc.index) + list(cash.index)
        if not available:
            return {"quarters": [], "revenue": [], "eps": [], "ocf": []}
        index = pd.period_range(min(available), max(available), freq="Q-DEC")
        inc, cash = inc.reindex(index), cash.reindex(index)

        def column(frame, names):
            result = pd.Series(float("nan"), index=index)
            for name in names:
                if name in frame:
                    result = result.combine_first(frame[name])
            return result

        revenue = column(inc, ["Revenue"])
        eps = column(inc, ["EPS"])
        ytd_ocf = column(cash, ["CashFlowsFromOperatingActivities", "NetCashInflowFromOperatingActivities"])
        ocf = ytd_ocf.diff()
        ocf.loc[index.quarter == 1] = ytd_ocf.loc[index.quarter == 1]

        def values(series):
            return [float(v) if pd.notna(v) and math.isfinite(v) else None for v in series.iloc[-4:][::-1]]

        return {
            "quarters": [str(q) for q in index[-4:][::-1]],
            "revenue": values(revenue), "eps": values(eps), "ocf": values(ocf),
        }

    def get_forward_eps(self, ticker: str) -> dict[str, Any]:
        """Forward EPS direct lookup; annual EBITDA supplementation is separately scoped.

        No Yahoo price, reported EPS, statements, or other valuation fields are
        copied into the FinMind snapshot. Missing data is never replaced by Nowcast.
        """
        from datetime import datetime, timezone
        result = {
            "value": None, "source": None, "checked_at": None,
            "symbol": None, "currency": None, "status": "Yahoo forwardEps unavailable",
        }
        code = self.ticker_code(ticker)
        symbol = str(ticker).strip().upper()
        if not symbol.endswith((".TW", ".TWO")):
            # Resolve bare codes from FinMind metadata, not by probing both Yahoo markets.
            info = self._request("TaiwanStockInfo")
            selected = info[info["stock_id"].astype(str) == code] if {"stock_id", "type"}.issubset(info.columns) else pd.DataFrame()
            if "date" in selected:
                selected = selected.sort_values("date")
            market = selected.iloc[-1]["type"] if not selected.empty else None
            suffix = {"twse": ".TW", "tpex": ".TWO"}.get(market)
            if suffix is None:
                result["status"] = "Unknown listing market; use .TW or .TWO"
                return result
            symbol = code + suffix
        result["symbol"] = symbol
        if symbol in self._forward_eps_cache:
            return self._forward_eps_cache[symbol].copy()
        result["checked_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
        try:
            import yfinance as yf
            info = yf.Ticker(symbol).get_info()
            if isinstance(info, dict):
                self._yahoo_info_cache[symbol] = info
            if not isinstance(info, dict):
                result["status"] = "Invalid Yahoo response"
            else:
                # Optional metadata must not block Yahoo's forwardEps value.
                result["currency"] = info.get("financialCurrency")
                value = info.get("forwardEps")
                if value is None or isinstance(value, bool):
                    result["status"] = "Yahoo forwardEps unavailable"
                else:
                    try:
                        value = float(value)
                    except (TypeError, ValueError):
                        value = float("nan")
                    if not math.isfinite(value):
                        result["status"] = "Yahoo forwardEps invalid"
                    else:
                        result["value"] = value
                        result["source"] = "Yahoo Finance (forwardEps)"
                        result["status"] = "Available" if value > 0 else "Non-positive EPS; Forward PE unavailable"
                        self._forward_eps_cache[symbol] = result.copy()
        except ImportError:
            result["status"] = "yfinance is not installed"
        except Exception as exc:
            # Never include exception URLs, responses, or credentials in a report.
            result["status"] = f"Yahoo request failed: {type(exc).__name__}"
        return result

    def get_analyst_consensus(self, ticker: str) -> dict[str, Any]:
        """Yahoo current-month analyst counts and aggregate price targets; reference only."""
        from datetime import datetime, timezone

        result = {
            "summary": "資料不足", "strong_buy": None, "buy": None,
            "hold": None, "sell": None, "strong_sell": None,
            "rating_count": None, "period": None,
            "target_low": None, "target_mean": None,
            "target_median": None, "target_high": None,
            "currency": None, "source": "Yahoo Finance (yfinance)",
            "symbol": None,
            "checked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "status": "資料不足",
        }
        try:
            code = self.ticker_code(ticker)
            symbol = str(ticker).strip().upper()
            if not symbol.endswith((".TW", ".TWO")):
                info = self._request("TaiwanStockInfo")
                selected = info[info["stock_id"].astype(str) == code] if {"stock_id", "type"}.issubset(info.columns) else pd.DataFrame()
                if "date" in selected:
                    selected = selected.sort_values("date")
                market = selected.iloc[-1]["type"] if not selected.empty else None
                suffix = {"twse": ".TW", "tpex": ".TWO"}.get(market)
                if suffix is None:
                    result["status"] = "無法判定Yahoo上市市場"
                    return result
                symbol = code + suffix
            result["symbol"] = symbol
        except Exception as exc:
            result["status"] = f"Yahoo代號判定失敗：{type(exc).__name__}"
            return result

        if symbol in self._analyst_consensus_cache:
            return self._analyst_consensus_cache[symbol].copy()

        errors = []
        try:
            import yfinance as yf
            instrument = yf.Ticker(symbol)
        except ImportError:
            result["status"] = "未安裝yfinance"
            return result
        except Exception as exc:
            result["status"] = f"Yahoo初始化失敗：{type(exc).__name__}"
            return result

        try:
            recommendations = instrument.get_recommendations()
            if isinstance(recommendations, pd.DataFrame) and not recommendations.empty:
                if "period" in recommendations.columns:
                    periods = recommendations["period"].astype(str)
                    current = recommendations.loc[periods.eq("0m")]
                    result["period"] = "0m" if not current.empty else str(recommendations.iloc[0].get("period"))
                else:
                    index_periods = pd.Index(recommendations.index).astype(str)
                    current = recommendations.loc[index_periods == "0m"]
                    result["period"] = "0m" if not current.empty else str(recommendations.index[0])
                if not current.empty:
                    row = current.iloc[0]
                    keys = ("strongBuy", "buy", "hold", "sell", "strongSell")
                    counts = []
                    for key in keys:
                        value = pd.to_numeric(pd.Series([row.get(key)]), errors="coerce").iloc[0]
                        counts.append(int(value) if pd.notna(value) and float(value) >= 0 else None)
                    if all(value is not None for value in counts):
                        (result["strong_buy"], result["buy"], result["hold"],
                         result["sell"], result["strong_sell"]) = counts
                        result["rating_count"] = sum(counts)
                        result["summary"] = "/".join(str(value) for value in counts)
                else:
                    errors.append("本月評等缺漏")
            else:
                errors.append("評等缺漏")
        except Exception as exc:
            errors.append(f"評等:{type(exc).__name__}")

        try:
            targets = instrument.get_analyst_price_targets()
            if isinstance(targets, dict):
                for result_key, yahoo_key in (("target_low", "low"), ("target_mean", "mean"),
                                              ("target_median", "median"), ("target_high", "high")):
                    value = pd.to_numeric(pd.Series([targets.get(yahoo_key)]), errors="coerce").iloc[0]
                    if pd.notna(value) and math.isfinite(float(value)) and float(value) > 0:
                        result[result_key] = float(value)
            if not any(result[key] is not None for key in ("target_low", "target_mean", "target_median", "target_high")):
                errors.append("目標價缺漏")
        except Exception as exc:
            errors.append(f"目標價:{type(exc).__name__}")

        info = self._yahoo_info_cache.get(symbol, {})
        if not info:
            try:
                info = instrument.get_info()
                if isinstance(info, dict):
                    self._yahoo_info_cache[symbol] = info
            except Exception:
                info = {}
        if isinstance(info, dict):
            result["currency"] = info.get("currency") or info.get("financialCurrency")
        ratings_ok = result["summary"] != "資料不足"
        targets_ok = any(result[key] is not None for key in ("target_low", "target_high"))
        result["status"] = "可用" if ratings_ok and targets_ok else "部分可用" if ratings_ok or targets_ok else "資料不足"
        if errors:
            result["status"] += "；" + "；".join(errors)
        self._analyst_consensus_cache[symbol] = result.copy()
        return result

    @staticmethod
    def _cashflow_quality(reference: dict[str, Any], annual_cashflow: pd.DataFrame, info: dict[str, Any]) -> dict[str, Any]:
        """Reference-only cash quality. FCF = OCF - nonnegative CapEx outflow.

        Mapped CapEx is expected to be zero or negative. Positive
        values are ambiguous and are not silently converted with abs().
        Annual positivity is persistence, not a measure of cash-flow volatility.
        """
        notes = list(reference.get("notes", []))

        def finite(value):
            try:
                value = float(value)
                return value if math.isfinite(value) else None
            except (TypeError, ValueError):
                return None

        def capex_outflow(value):
            value = finite(value)
            return -value if value is not None and value <= 0 else None

        def total(values):
            clean = [finite(value) for value in values]
            return float(sum(clean)) if len(clean) == 4 and all(value is not None for value in clean) else None

        result = {
            "currency": info.get("financialCurrency"),
            "period": " / ".join(reversed(reference.get("quarters", []))),
            "ocf_ttm": total(reference.get("ocf", [])),
            "net_income_ttm": total(reference.get("net_income", [])),
            "revenue_ttm": total(reference.get("revenue", [])),
            "capex_ttm": total([capex_outflow(v) for v in reference.get("capex", [])]),
            "fcf_ttm": None, "ocf_to_net_income": None, "ocf_quality_status": "資料不足",
            "fcf_margin_ttm": None, "fcf_yield_ttm": None,
            "fcf_positive_ratio_5y": None, "fcf_positive_ratio_available": None,
            "fcf_positive_years": 0, "fcf_valid_years": 0,
            "fcf_positive_streak": None, "fcf_streak_status": "資料不足",
            "annual_fcf": [], "notes": notes,
        }
        if result["ocf_ttm"] is not None and result["capex_ttm"] is not None:
            result["fcf_ttm"] = result["ocf_ttm"] - result["capex_ttm"]
        profit = result["net_income_ttm"]
        if profit is not None and profit <= 0:
            result["ocf_quality_status"] = "淨利非正，不適用一般比率級距"
        elif profit is not None and result["ocf_ttm"] is not None:
            ratio = safe_divide(result["ocf_ttm"], profit)
            result["ocf_to_net_income"] = ratio
            if ratio is not None:
                result["ocf_quality_status"] = "高於1" if ratio > 1 else "0.8至1" if ratio >= .8 else "0.5至0.8以下" if ratio >= .5 else "低於0.5"
        revenue = result["revenue_ttm"]
        if revenue is not None and revenue > 0:
            result["fcf_margin_ttm"] = safe_divide(result["fcf_ttm"], revenue)
        market_cap = finite(info.get("marketCap"))
        if result["currency"] and result["currency"] == info.get("currency"):
            if market_cap is not None and market_cap > 0:
                result["fcf_yield_ttm"] = safe_divide(result["fcf_ttm"], market_cap)
        else:
            notes.append("財報與報價幣別不同或不明：TTM FCF殖利率留空，不自動換匯")
        for field in ("ocf_ttm", "net_income_ttm", "revenue_ttm", "capex_ttm"):
            if result[field] is None:
                notes.append(f"{field}: 最近四季資料不完整或數值不適用")
        if any(finite(v) is not None and finite(v) > 0 for v in reference.get("capex", [])):
            notes.append("季度 Capital Expenditure 為正值：支出口徑不明，未取絕對值")

        if not isinstance(annual_cashflow, pd.DataFrame) or annual_cashflow.empty:
            notes.append("年度現金流缺漏，無法評估FCF持續性")
            return result
        frame = annual_cashflow.copy()
        frame.columns = pd.to_datetime(frame.columns, errors="coerce", utc=True).tz_convert(None).normalize()
        frame = frame.loc[:, ~frame.columns.isna()]
        if frame.columns.duplicated().any():
            notes.append("年度現金流日期重複：排除歧義日期")
            frame = frame.loc[:, ~frame.columns.duplicated(keep=False)]
        if frame.index.duplicated().any():
            notes.append("年度現金流項目重複：排除歧義項目")
            frame = frame.loc[~frame.index.duplicated(keep=False)]
        dates = sorted(frame.columns, reverse=True)
        if not dates:
            return result
        for offset in range(5):
            expected = dates[0] - pd.DateOffset(years=offset)
            matches = [d for d in dates if abs((d - expected).days) <= 15]
            endpoint = matches[0] if len(matches) == 1 else None
            ocf = capex = None
            if endpoint is not None:
                if "Operating Cash Flow" in frame.index:
                    ocf = finite(frame.at["Operating Cash Flow", endpoint])
                if "Capital Expenditure" in frame.index:
                    raw = frame.at["Capital Expenditure", endpoint]
                    capex = capex_outflow(raw)
                    if finite(raw) is not None and finite(raw) > 0:
                        notes.append(f"年度 {endpoint.date()}: Capital Expenditure 為正值，FCF留空")
            fcf = ocf - capex if ocf is not None and capex is not None else None
            result["annual_fcf"].append({
                "period_end": endpoint.date().isoformat() if endpoint is not None else None,
                "expected_period_end": expected.date().isoformat(),
                "ocf": ocf, "capex": capex, "fcf": fcf,
            })
        values = [row["fcf"] for row in result["annual_fcf"]]
        valid = [v for v in values if v is not None]
        positive = sum(v > 0 for v in valid)
        result["fcf_positive_years"] = positive
        result["fcf_valid_years"] = len(valid)
        result["fcf_positive_ratio_available"] = safe_divide(positive, len(valid))
        if len(valid) == 5:
            result["fcf_positive_ratio_5y"] = positive / 5
        else:
            notes.append(f"五年度窗口僅 {len(valid)} 年有效，完整五年FCF正值比例留空")
        if values[0] is not None:
            streak = 0
            for value in values:
                if value is None or value <= 0:
                    break
                streak += 1
            result["fcf_positive_streak"] = streak
            result["fcf_streak_status"] = (
                "至少5年（僅檢查五年度窗口）" if streak == 5 else
                f"至少{streak}年；較早年度缺漏" if values[streak] is None else
                f"連續{streak}年；遇非正值年度停止"
            )
        return result

    def get_fundamental_review_bundle(self, ticker, snapshot, info):
        """FinMind-only supplementary inputs; never overwrite old snapshot fields."""
        notes=[]
        def table(raw):
            if raw is None or raw.empty or not {'date','type','value'}.issubset(raw.columns):
                return pd.DataFrame()
            data=raw.copy()
            data['date']=pd.to_datetime(data['date'],errors='coerce')
            data['value']=pd.to_numeric(data['value'],errors='coerce')
            data=data.dropna(subset=['date','type'])
            data['quarter']=data['date'].dt.to_period('Q-DEC')
            # Conflicting duplicate values are unavailable, never arbitrarily picked.
            series=data.groupby(['quarter','type'])['value'].agg(lambda s:s.iloc[0] if s.nunique(dropna=False)==1 else float('nan'))
            return series.unstack('type').sort_index()
        inc=table(self.get_financial_statements(ticker))
        cash=table(self.get_cash_flow_statement(ticker))
        bal=table(self.get_balance_sheet(ticker))
        available=list(inc.index)+list(cash.index)
        def number(v):
            try:
                v=float(v)
                return v if math.isfinite(v) else None
            except (ValueError,TypeError):return None
        def value(frame,q,names):
            if q not in frame.index:return None
            for key in names:
                if key in frame:
                    v=number(frame.loc[q,key])
                    if v is not None:return v
            return None
        def ytd_quarter(frame,q,names):
            current=value(frame,q,names)
            if q.quarter==1:return current
            previous=value(frame,q-1,names)
            return current-previous if current is not None and previous is not None else None
        ocf_names=['CashFlowsFromOperatingActivities','NetCashInflowFromOperatingActivities']
        capex_names=['PropertyAndPlantAndEquipment']
        end=max(available) if available else None
        quarters=list(pd.period_range(end=end,periods=4,freq='Q-DEC')[::-1]) if end is not None else []
        reference={
            'quarters':[q.end_time.date().isoformat() for q in quarters],
            'revenue':[value(inc,q,['Revenue']) for q in quarters],
            'net_income':[value(inc,q,['IncomeAfterTaxes','NetIncome']) for q in quarters],
            'ocf':[ytd_quarter(cash,q,ocf_names) for q in quarters],
            'capex':[ytd_quarter(cash,q,capex_names) for q in quarters],
            'notes':['FinMind現金流YTD轉單季；CapEx僅採購置不動產廠房設備，未包含未映射的無形資產支出'],
        }
        annual={}
        for q in cash.index:
            if q.quarter==4:
                annual[pd.Timestamp(q.end_time.date())]={'Operating Cash Flow':value(cash,q,ocf_names),'Capital Expenditure':value(cash,q,capex_names)}
        cf=self._cashflow_quality(reference,pd.DataFrame(annual),{'financialCurrency':'TWD','currency':'TWD','marketCap':snapshot.market_cap})
        category=None
        if info is not None and {'stock_id','industry_category'}.issubset(info.columns):
            selected=info[info['stock_id'].astype(str)==self.ticker_code(ticker)]
            if 'date' in selected:selected=selected.sort_values('date')
            if not selected.empty and pd.notna(selected.iloc[-1]['industry_category']):category=str(selected.iloc[-1]['industry_category'])
        inputs={'sector':category,'industry':category,'asset_type':snapshot.asset_type,'currency':'TWD','period':None,
                'source':'FinMind同年度損益表＋年末資產負債表','note':'', 'debt_basis':None}
        if category and any(word in category.upper() for word in ('ETF','ETN','受益憑證','受益證券','基金')):
            inputs['asset_type']='FUND'
        if inc.empty:
            inputs['note']='FinMind損益表缺漏'
            return cf,inputs
        latest=max(inc.index)
        year=latest.year if latest.quarter==4 else latest.year-1
        q4=pd.Period(f'{year}Q4',freq='Q-DEC')
        year_quarters=list(pd.period_range(f'{year}Q1',f'{year}Q4',freq='Q-DEC'))
        if not all(q in inc.index for q in year_quarters) or q4 not in bal.index:
            inputs['note']='最近完整年度四季損益或同年末資產負債表缺漏，不回退拼接不同年度'
            return cf,inputs
        inputs['period']=q4.end_time.date().isoformat()
        def annual_sum(names):
            values=[value(inc,q,names) for q in year_quarters]
            return sum(values) if all(v is not None for v in values) else None
        inputs.update(review_annual_revenue=annual_sum(['Revenue']),
                      review_annual_net_income=annual_sum(['IncomeAfterTaxes','NetIncome']),
                      operating_income=annual_sum(['OperatingIncome']),
                      interest=annual_sum(['InterestExpense']),
                      ebitda=annual_sum(['EBITDA']),
                      cash=value(bal,q4,['CashAndCashEquivalents']),cash_row='CashAndCashEquivalents',
                      current_assets=value(bal,q4,['CurrentAssets']),
                      current_liabilities=value(bal,q4,['CurrentLiabilities']))
        inputs['debt']=value(bal,q4,['TotalDebt'])
        if inputs['debt'] is not None:
            inputs['debt_basis']='FinMind TotalDebt'
        else:
            parts=[value(bal,q4,names) for names in (
                ['ShorttermBorrowings','ShortTermBorrowings'],['LongtermBorrowings','LongTermBorrowings'],['BondsPayable'])]
            inputs['debt']=sum(parts) if all(v is not None and v>=0 for v in parts) else None
            inputs['debt_basis']='短期借款＋長期借款＋公司債；可能未涵蓋其他借款／租賃，不據此評為穩健'
            inputs['debt_partial']=True
        inputs['ebitda_source']='FinMind' if inputs.get('ebitda') is not None else None
        inputs['ebitda_status']='FinMind已有值' if inputs.get('ebitda') is not None else 'FinMind缺值，將檢查整組Yahoo財務安全備援'
        inputs['note']='FinMind不足時財務安全性整組採用Yahoo同年度財報；ROIC另依缺值情況補算'
        return cf,inputs


    def _yahoo_annual_financial_support(self, ticker):
        """One Yahoo fiscal-year bundle, used only for safety and missing ROIC."""
        from datetime import datetime, timezone
        result={'period':None,'source':'Yahoo年度損益表＋資產負債表','note':'',
                'checked_at':datetime.now(timezone.utc).isoformat(timespec='seconds')}
        try:
            forward=self.get_forward_eps(ticker)
            symbol=forward.get('symbol')
            result['symbol']=symbol
            if not symbol:
                result['note']='無法確認Yahoo上市／上櫃代號'
                return result
            cache=getattr(self,'_yahoo_annual_support_cache',{})
            if symbol in cache:return cache[symbol].copy()
            import yfinance as yf
            instrument=yf.Ticker(symbol)
            info=getattr(self,'_yahoo_info_cache',{}).get(symbol)
            if not isinstance(info,dict):info=instrument.get_info()
            if not isinstance(info,dict) or str(info.get('symbol',symbol)).upper()!=symbol:
                result['note']='Yahoo公司識別資料不一致'
                return result
            result.update(sector=info.get('sector'),industry=info.get('industry'),
                          asset_type=info.get('quoteType'),currency=info.get('financialCurrency'))
            if not result['currency']:
                result['note']='Yahoo財報幣別不明，不計算跨報表比率'
                return result
            income=instrument.get_income_stmt(pretty=True,freq='yearly')
            balance=instrument.get_balance_sheet(pretty=True,freq='yearly')
            if not all(isinstance(frame,pd.DataFrame) and not frame.empty for frame in (income,balance)):
                result['note']='Yahoo年度損益表或資產負債表缺漏'
                return result
            if income.index.duplicated().any() or balance.index.duplicated().any():
                result['note']='Yahoo年度資料存在重複項目，無法確定口徑'
                return result
            income_dates={pd.Timestamp(c).date():c for c in income.columns}
            balance_dates={pd.Timestamp(c).date():c for c in balance.columns}
            if len(income_dates)!=len(income.columns) or len(balance_dates)!=len(balance.columns):
                result['note']='Yahoo年度日期重複'
                return result
            latest=max(income_dates)
            if latest!=max(balance_dates):
                result['note']='Yahoo兩張最新年度財報期末不同，不混期計算'
                return result
            if not 0 <= (date.today()-latest).days <= 550:
                result['note']='Yahoo年度財報超過550天或日期異常'
                return result
            result['period']=latest.isoformat()
            def cell(frame,column,labels):
                for label in labels:
                    if label in frame.index:
                        value=IndependentInvestmentAdvice._number(frame.loc[label,column])
                        if value is not None:return value,label
                return None,None
            for key,labels in {
                'debt':['Total Debt'],
                'cash':['Cash And Cash Equivalents','Cash Cash Equivalents And Short Term Investments'],
                'equity':['Stockholders Equity','Total Equity Gross Minority Interest'],
                'current_assets':['Current Assets'], 'current_liabilities':['Current Liabilities'],
            }.items():result[key],result[key+'_row']=cell(balance,balance_dates[latest],labels)
            result['debt_basis']='Yahoo Total Debt'
            if result['debt'] is None:
                short,shortrow=cell(balance,balance_dates[latest],['Current Debt And Capital Lease Obligation'])
                long,longrow=cell(balance,balance_dates[latest],['Long Term Debt And Capital Lease Obligation'])
                if short is not None and long is not None and short>=0 and long>=0:
                    result['debt']=short+long
                    result['debt_basis']='Yahoo流動＋非流動借款及資本租賃負債'
                else:result['debt_basis']='Yahoo Total Debt與完整借款組成都缺漏'
            for key,labels in {
                'operating_income':['Operating Income'],
                'interest':['Interest Expense','Interest Expense Non Operating'],
                'ebitda':['EBITDA'],
            }.items():result[key],result[key+'_row']=cell(income,income_dates[latest],labels)
            result.update(debt_partial=False,ebitda_source='Yahoo年度財報',
                          ebitda_status='財務安全性整組採用Yahoo同年度財報，不混FinMind金額',
                          ebitda_checked_at=result['checked_at'],ebitda_yahoo_symbol=symbol,
                          ebitda_currency=result['currency'],ebitda_yahoo_raw=result['ebitda'],
                          ebitda_unit_scale=None,ebitda_unit_checks='同組資料全為Yahoo原始單位；不與FinMind負債相除，無需跨來源倍率')
            result['note']='整組Yahoo同年度同幣別；不採info中的TTM負債／EBITDA或即時比率'
            cache[symbol]=result.copy();self._yahoo_annual_support_cache=cache
        except Exception as exc:
            result['period']=None
            result['note']=f'Yahoo財報備援失敗：{type(exc).__name__}'
        return result

    def supplement_safety_and_roic(self, ticker, snapshot):
        finite=IndependentInvestmentAdvice._number
        original=snapshot.fundamental_review_inputs
        snapshot.cash_review_scope=original.copy()
        original_safety=FundamentalRiskReviews.safety(original)
        scope_status,_=FundamentalRiskReviews.applicability(original)
        needs_safety=(original_safety['label']=='資料不足' or
                      (original_safety['label']=='注意' and original.get('debt_partial')))
        op=finite(snapshot.operating_income);equity=finite(snapshot.total_equity)
        debt=finite(snapshot.total_debt);cash=finite(snapshot.cash)
        capital=equity+debt-cash if all(v is not None for v in (equity,debt,cash)) else None
        native=finite(safe_divide(op*.8 if op is not None else None,capital))
        snapshot.roic_meta={'source':'FinMind' if native is not None else None,
                            'status':'沿用原FinMind計算' if native is not None else 'FinMind無法計算',
                            'period':None,'currency':'TWD','checked_at':None,
                            'operating_income':op,'equity':equity,'debt':debt,'cash':cash,
                            'invested_capital':capital,'assumed_tax_rate':.2,
                            'formula':'營業利益×(1−20%)／(權益＋有息負債−現金)；保留既有期末資本近似法'}
        snapshot.safety_fallback_meta={'status':'保留FinMind','original_status':original_safety['label'],
                                       'original_note':original_safety['note'],'note':'','period':None,'checked_at':None}
        if scope_status=='不適用':
            snapshot.safety_fallback_meta['status']='不適用一般企業規則，未向Yahoo補財報'
            if native is None:snapshot.roic_meta['status']='不適用一般企業ROIC備援'
            return
        if not needs_safety and native is not None:return
        bundle=self._yahoo_annual_financial_support(ticker)
        snapshot.safety_fallback_meta.update(note=bundle.get('note'),period=bundle.get('period'),checked_at=bundle.get('checked_at'))
        # Industry metadata remains FinMind-first; Yahoo metadata fills only missing scope labels.
        for key in ('sector','industry','asset_type'):
            if original.get(key):bundle[key]=original[key]
        bundle_scope,_=FundamentalRiskReviews.applicability(bundle)
        if not bundle.get('period') or bundle_scope:
            if needs_safety:snapshot.safety_fallback_meta['status']='Yahoo備援未取得可用且適用的同年度資料'
            if native is None:snapshot.roic_meta['status']=bundle.get('note') or 'Yahoo資料不適用或缺少分類'
            return
        if needs_safety:
            yahoo_safety=FundamentalRiskReviews.safety(bundle)
            if yahoo_safety['label']!='資料不足':
                snapshot.fundamental_review_inputs=bundle.copy()
                snapshot.safety_fallback_meta['status']='財務安全性已改用Yahoo同年度財報'
                snapshot.source+=' + Yahoo financial safety'
            else:
                # Show the more complete single-source set; never merge financial amounts.
                keys=('debt','cash','operating_income','interest','ebitda','current_assets','current_liabilities')
                known=lambda values:sum(finite(values.get(k)) is not None for k in keys)
                if known(bundle)>known(original):
                    snapshot.fundamental_review_inputs=bundle.copy()
                    snapshot.source+=' + Yahoo financial safety (partial)'
                snapshot.safety_fallback_meta['status']='已查Yahoo，但仍缺必要財務資料'
                snapshot.safety_fallback_meta['note']=yahoo_safety['note']
        if native is None:
            vals={key:finite(bundle.get(key)) for key in ('operating_income','equity','debt','cash')}
            invested=vals['equity']+vals['debt']-vals['cash'] if all(v is not None for v in vals.values()) else None
            snapshot.roic_meta.update(source='Yahoo年度財報計算',period=bundle['period'],currency=bundle.get('currency'),
                                      checked_at=bundle.get('checked_at'),invested_capital=invested,**vals)
            if invested is None or invested<=0 or vals['debt']<0 or vals['cash']<0:
                snapshot.roic_meta['status']='Yahoo ROIC所需資料不足或投入資本非正，留空'
            else:
                value=finite(vals['operating_income']*.8/invested)
                snapshot.roic_fallback=value
                snapshot.roic_meta['status']='Yahoo同年度財報補算完成' if value is not None else 'Yahoo ROIC無有效值'
                if value is not None:snapshot.source+=' + Yahoo ROIC'


In [ ]:
# 品質評估分數定義
# =========================
# Quality
# =========================

class QualityEngine:
    def calculate(self, snapshot: FinancialSnapshot) -> QualityMetrics:
        invested_capital = None
        if snapshot.total_equity is not None and snapshot.total_debt is not None and snapshot.cash is not None:
            invested_capital = snapshot.total_equity + snapshot.total_debt - snapshot.cash

        nopat = snapshot.operating_income * 0.8 if snapshot.operating_income is not None else None
        native_roic = safe_divide(nopat, invested_capital)
        roic = native_roic if native_roic is not None and math.isfinite(native_roic) else snapshot.roic_fallback
        history = snapshot.financial_history

        return QualityMetrics(
            roe=safe_divide(snapshot.net_income, snapshot.total_equity),
            roic=roic,
            gross_margin=safe_divide(snapshot.gross_profit, snapshot.revenue),
            operating_margin=safe_divide(snapshot.operating_income, snapshot.revenue),
            net_margin=safe_divide(snapshot.net_income, snapshot.revenue),
            debt_to_equity=safe_divide(snapshot.total_debt, snapshot.total_equity),
            interest_coverage=safe_divide(snapshot.operating_income, snapshot.interest_expense),
            current_ratio=safe_divide(snapshot.current_assets, snapshot.current_liabilities),
            quick_ratio=safe_divide(
                (snapshot.current_assets - snapshot.inventory)
                if snapshot.current_assets is not None and snapshot.inventory is not None
                else None,
                snapshot.current_liabilities,
            ),
            fcf_margin=safe_divide(snapshot.free_cash_flow, snapshot.revenue),
            revenue_cagr_1y=self._cagr(history.get("revenue", []), 1),
            revenue_cagr_3y=self._cagr(history.get("revenue", []), 3),
            revenue_cagr_5y=self._cagr(history.get("revenue", []), 5),
            revenue_cagr_10y=self._cagr(history.get("revenue", []), 10),
            eps_cagr_1y=self._cagr(history.get("eps", []), 1),
            eps_cagr_3y=self._cagr(history.get("eps", []), 3),
            eps_cagr_5y=self._cagr(history.get("eps", []), 5),
            eps_cagr_10y=self._cagr(history.get("eps", []), 10),
            fcf_cagr_1y=self._cagr(history.get("free_cash_flow", []), 1),
            fcf_cagr_3y=self._cagr(history.get("free_cash_flow", []), 3),
            fcf_cagr_5y=self._cagr(history.get("free_cash_flow", []), 5),
            fcf_cagr_10y=self._cagr(history.get("free_cash_flow", []), 10),
            **self._reference_metrics(snapshot),
        )

    @staticmethod
    def _cagr(values: list[float], years: int) -> float | None:
        if len(values) <= years or values[years] <= 0 or values[0] <= 0:
            return None
        return (values[0] / values[years]) ** (1 / years) - 1

    @staticmethod
    def _period_change(values: list[float | None], periods: int) -> tuple[float | None, str]:
        """1 => q0/q1-1; 2 => q1/q2-1; 3 => q2/q3-1. Reference only."""
        if periods < 1 or len(values) <= periods:
            return None, "資料不足"
        current, previous = values[periods - 1], values[periods]
        if current is None or previous is None or not math.isfinite(current) or not math.isfinite(previous):
            return None, "資料不足"
        if previous < 0:
            status = "由負轉正" if current > 0 else "由負轉零" if current == 0 else "負值改善" if current > previous else "負值惡化" if current < previous else "負值持平"
            return None, status
        if previous == 0:
            return None, "由零轉正" if current > 0 else "由零轉負" if current < 0 else "零值持平"
        rate = safe_divide(current, previous)
        if rate is None or not math.isfinite(rate - 1):
            return None, "比率無法計算"
        status = "由正轉負" if current < 0 else "成長" if current > previous else "衰退" if current < previous else "持平"
        return rate - 1, status

    @classmethod
    def _period_growth(cls, values: list[float | None], periods: int) -> float | None:
        return cls._period_change(values, periods)[0]

    @classmethod
    def _reference_metrics(cls, snapshot: FinancialSnapshot) -> dict[str, Any]:
        reference = snapshot.quarterly_reference
        result = {"growth_status": {}, "growth_periods": {}}
        for metric in ("revenue", "eps", "ocf"):
            values = reference.get(metric, [])
            for step, months in enumerate((3, 6, 9), start=1):
                name = f"{metric}_growth_{months}m"
                rate, status = cls._period_change(values, step)
                result[name] = rate
                result["growth_status"][name] = status
        quarters = reference.get("quarters", [])
        for step, months in enumerate((3, 6, 9), start=1):
            result["growth_periods"][f"{months}m"] = f"{quarters[step-1]} / {quarters[step]}" if len(quarters) > step else "資料不足"
        return result


# =========================
# Decision / scoring
# =========================


class DecisionEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.thresholds = config["scoring"]["recommendations"]

    def decide(self, score: float | None) -> Recommendation:
        if score is None:
            return Recommendation("Insufficient Data", "☆☆☆☆☆", None)
        if score >= self.thresholds["strong_buy"]:
            return Recommendation("Strong Buy", "★★★★★", score)
        if score >= self.thresholds["buy"]:
            return Recommendation("Buy", "★★★★☆", score)
        if score >= self.thresholds["hold"]:
            return Recommendation("Hold", "★★★☆☆", score)
        if score >= self.thresholds["reduce"]:
            return Recommendation("Reduce", "★★☆☆☆", score)
        return Recommendation("Avoid", "★☆☆☆☆", score)


class ScoringEngine:
    REFERENCE_ONLY = frozenset(
        f"{metric}_growth_{months}m"
        for metric in ("revenue", "eps", "ocf") for months in (3, 6, 9)
    )
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config
        
        
    
    

    def quality_score(self, metrics: Any, fcf_yield: float | None) -> float | None:
        data = asdict(metrics)
        data["fcf_yield"] = fcf_yield

        total = 0.0
        available_weight = 0.0

        for name, rule in self.config["quality_score"]["metrics"].items():
            if name in self.REFERENCE_ONLY:
                continue
            value = data.get(name)
            if value is None:
                continue

            weight = rule["weight"]
            excellent = rule["excellent"]
            good = rule["good"]
            lower = rule.get("lower_is_better", False)

            if lower:
                points = 1.0 if value <= excellent else 0.6 if value <= good else 0.2
            else:
                points = 1.0 if value >= excellent else 0.6 if value >= good else 0.2

            total += weight * points
            available_weight += weight

        return 100 * total / available_weight if available_weight else None
    
    
    
    





    def growth_score(self, metrics: Any) -> float | None:
        rules = self.config.get("growth_score", {}).get("metrics", {})
        if not rules:
            return None

        total = 0.0
        available_weight = 0.0
        for name, rule in rules.items():
            if name in self.REFERENCE_ONLY:
                continue
            value = getattr(metrics, name, None)
            if value is None:
                continue

            weight = rule["weight"]
            excellent = rule["excellent"]
            good = rule["good"]
            lower = rule.get("lower_is_better", False)
            if lower:
                points = 1.0 if value <= excellent else 0.6 if value <= good else 0.2
            else:
                points = 1.0 if value >= excellent else 0.6 if value >= good else 0.2
            total += weight * points
            available_weight += weight

        return 100 * total / available_weight if available_weight else None
    
        #買賣時機分數  
    def technical_score(
        self,
        technical,
        price
    ):

        score = 0
        total_weight = 0


        # ==========================
        # 1. RSI (35%)
        # 越低越有買點價值
        # ==========================

        if technical.rsi is not None:

            total_weight += 35

            rsi = technical.rsi

            if rsi < 30:
                score += 35

            elif rsi < 40:
                score += 30

            elif rsi <=50:
                score += 25

            elif rsi <=60:
                score += 15

            elif rsi <=70:
                score += 5

            else:
                score += 0



        # ==========================
        # 2. 支撐距離 (35%)
        # 越靠近支撐越好
        # ==========================

        if technical.support is not None:

            total_weight +=35

            distance = (
                price - technical.support
            ) / price


            if distance <=0.02:
                score +=35

            elif distance <=0.05:
                score +=30

            elif distance <=0.10:
                score +=20

            else:
                score +=5



        # ==========================
        # 3. 壓力距離 (20%)
        # 避免追高
        # ==========================

        if technical.resistance is not None:

            total_weight +=20

            distance = (
                technical.resistance-price
            ) / price


            # 還有很大上漲空間
            if distance >=0.15:
                score +=20

            elif distance >=0.08:
                score +=15

            elif distance >=0.03:
                score +=8

            else:
                score +=0



        # ==========================
        # 4. 均線趨勢 (10%)
        # 趨勢只輔助
        # ==========================

        if (
            technical.short_ma is not None
            and technical.long_ma is not None
        ):

            total_weight +=10


            if technical.short_ma > technical.long_ma:
                score +=10

            elif technical.short_ma >= technical.long_ma*0.97:
                score +=5

            else:
                score +=0



        return (
            None
            if total_weight==0
            else score / total_weight *100
        )
    
    #籌碼面分數
    def flow_score(
        self,
        flow_signal: str | None,
        volume_ratio: float | None = None,
        return_20d: float | None = None,
    ) -> float | None:

        if flow_signal in {"Unavailable", "Insufficient Data"}:
            flow_signal = None


        score = 0
        total = 0


        # =====================
        # 資金訊號
        # =====================

        if flow_signal is not None:

            total += 40

            mapping = {

                "Strong Buy": 40,
                "Buy": 30,
                "Neutral": 20,
                "Sell": 10,
                "Sell/Reduce": 10,
                "Strong Sell": 0,

                # 中文防呆
                "強買":40,
                "買進":30,
                "中性":20,
                "賣出":10,
                "強賣":0,
            }

            score += mapping.get(
                flow_signal,
                20
            )


        # =====================
        # 成交量
        # =====================

        if volume_ratio is not None:

            total += 30


            if volume_ratio >= 1.5:
                score += 30

            elif volume_ratio >= 1.2:
                score += 20

            elif volume_ratio >= 1:
                score += 10

            else:
                score += 5



        # =====================
        # 20日報酬
        # =====================

        if return_20d is not None:

            total += 30


            if return_20d >= 0.10:
                score += 30

            elif return_20d >= 0.05:
                score += 20

            elif return_20d >= 0:
                score += 10

            else:
                score += 5



        # =====================
        # 防呆
        # =====================

        if total == 0:
            return None


        return score / total * 100
    

    def total_score(
        self,
        quality: float | None,
        growth: float | None,
        buffett: float | None,
    ) -> float | None:
        # Historical valuation percentiles are context only and never enter this score.
        return calculate_fundamental_score(
            quality=quality,
            growth=growth,
            buffett=buffett,
            weights=self.config["scoring"]["weights"],
        )

class BuffettChecklist:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["buffett"]

    def evaluate(self, metrics: Any, snapshot: Any) -> dict[str, bool | None]:
        return {
            "roic": self._compare(metrics.roic, self.rules.get("roic_min"), True),
            "roe": self._compare(metrics.roe, self.rules.get("roe_min"), True),
            "debt_to_equity": self._compare(metrics.debt_to_equity, self.rules.get("debt_to_equity_max"), False),
            "fcf_positive": None if snapshot.free_cash_flow is None else snapshot.free_cash_flow > 0,
            "eps_growth_positive": None if metrics.eps_cagr_3y is None else metrics.eps_cagr_3y > 0,
            "operating_margin_positive": None if metrics.operating_margin is None else metrics.operating_margin > 0,
            "share_count_not_increasing": None,
        }

    @staticmethod
    def _compare(value: float | None, threshold: float | None, greater: bool) -> bool | None:
        if value is None or threshold is None:
            return None
        return value >= threshold if greater else value <= threshold

    @staticmethod
    def score(checks: dict[str, bool | None]) -> float | None:
        available = [value for value in checks.values() if value is not None]
        return None if not available else 100 * sum(available) / len(available)

class QuarterlyTrendReference:
    """Chronological text only; never produces a score."""
    @staticmethod
    def analyze(metrics: QualityMetrics) -> dict[str, str]:
        trends = {}
        for metric in ("eps", "revenue", "ocf"):
            states = [metrics.growth_status.get(f"{metric}_growth_{m}m", "資料不足") for m in (9, 6, 3)]
            trends[f"{metric}_direction"] = " → ".join(states)
        return trends


In [ ]:
# 估值&定價系統
# =========================
# Valuation
# =========================
        
        
class ValuationEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config["valuation"]

    def calculate(
        self,
        snapshot: FinancialSnapshot,
        price_history: pd.DataFrame,
        historical_fundamentals: pd.DataFrame | None = None,
    ) -> ValuationMetrics:
        # Historical PE uses trailing twelve-month EPS, so Current PE must use the
        # same basis. Non-positive EPS cannot form a meaningful PE.
        current_pe = (
            safe_divide(snapshot.current_price, snapshot.ttm_eps)
            if snapshot.ttm_eps is not None and snapshot.ttm_eps > 0
            else None
        )
        forward_pe = (
            safe_divide(snapshot.current_price, snapshot.forward_eps)
            if snapshot.forward_eps is not None and snapshot.forward_eps > 0
            else None
        )
        pb = safe_divide(snapshot.current_price, snapshot.book_value_per_share)
        growth = self._cagr(snapshot.financial_history.get("eps", []), 3)
        historical_ratios = self._point_in_time_ratios(price_history, historical_fundamentals)
        historical_pe_context = calculate_historical_pe_context(historical_ratios, current_pe)
        historical_pe_window_stats = calculate_historical_ratio_windows(historical_ratios, current_pe, "pe")
        historical_pb_window_stats = calculate_historical_ratio_windows(historical_ratios, pb, "pb")

        pe_series = historical_ratios["pe"].dropna().tolist() if not historical_ratios.empty else []
        pb_series = historical_ratios["pb"].dropna().tolist() if not historical_ratios.empty else []
        pe_dates = (
            historical_ratios.loc[historical_ratios["pe"].notna(), "date"]
            if not historical_ratios.empty
            else pd.Series(dtype="datetime64[ns]")
        )
        pe_history_years = (pe_dates.max() - pe_dates.min()).days / 365.25 if len(pe_dates) > 1 else None

        return ValuationMetrics(
            pe=current_pe,
            forward_pe=forward_pe,
            pb=pb,
            peg=safe_divide(current_pe, growth) if growth and growth > 0 else None,
            fcf_yield=safe_divide(snapshot.free_cash_flow, snapshot.market_cap),
            ev_ebit=safe_divide(snapshot.enterprise_value, snapshot.operating_income),
            dcf_value_per_share=self._dcf(snapshot),
            historical_pe_window_stats=historical_pe_window_stats,
            historical_pb_window_stats=historical_pb_window_stats,
            historical_pe_median_1y=historical_pe_context["historical_pe_median_1y"],
            historical_pe_percentile_1y=historical_pe_context["historical_pe_percentile_1y"],
            historical_pe_median_3y=historical_pe_context["historical_pe_median_3y"],
            historical_pe_percentile_3y=historical_pe_context["historical_pe_percentile_3y"],
            historical_pe_median_5y=historical_pe_context["historical_pe_median_5y"],
            historical_pe_percentile_5y=historical_pe_context["historical_pe_percentile_5y"],
            pe_median_trend_1y_vs_3y=historical_pe_context["pe_median_trend_1y_vs_3y"],
            pe_median_trend_1y_vs_5y=historical_pe_context["pe_median_trend_1y_vs_5y"],
            historical_pe_sample_count_1y=historical_pe_context["historical_pe_sample_count_1y"],
            historical_pe_sample_count_3y=historical_pe_context["historical_pe_sample_count_3y"],
            historical_pe_sample_count_5y=historical_pe_context["historical_pe_sample_count_5y"],
            pb_percentile_5y=percentile_rank(pb, pb_series),
            pe_sample_count=len(pe_series),
            pe_history_years=pe_history_years,
        )

    def _point_in_time_ratios(self, history: pd.DataFrame, fundamentals: pd.DataFrame | None) -> pd.DataFrame:
        if history is None or history.empty or fundamentals is None or fundamentals.empty or "Close" not in history:
            return pd.DataFrame(columns=["date", "pe", "pb"])

        prices = history[["Close"]].copy().sort_index()
        prices.index = pd.to_datetime(prices.index, utc=True).tz_convert(None)
        monthly_prices = prices["Close"].resample("ME").last().dropna().rename("close").reset_index()
        monthly_prices.columns = ["date", "close"]

        observations = fundamentals.copy()
        lag_days = self.config.get("earnings_availability_lag_days", 45)
        observations["effective_date"] = pd.to_datetime(observations["report_date"], utc=True).dt.tz_convert(None) + pd.DateOffset(days=lag_days)
        observations = observations.sort_values(["effective_date", "report_date"]).drop_duplicates("effective_date", keep="last")

        merged = pd.merge_asof(
            monthly_prices.sort_values("date"),
            observations[["effective_date", "ttm_eps", "bvps"]],
            left_on="date",
            right_on="effective_date",
            direction="backward",
        )
        merged["pe"] = [safe_divide(close, eps) if pd.notna(eps) and eps > 0 else None for close, eps in zip(merged["close"], merged["ttm_eps"])]
        merged["pb"] = [safe_divide(close, bvps) if pd.notna(bvps) and bvps > 0 else None for close, bvps in zip(merged["close"], merged["bvps"])]
        return merged[["date", "pe", "pb"]].dropna(how="all", subset=["pe", "pb"]).reset_index(drop=True)

    def _dcf(self, snapshot: FinancialSnapshot) -> float | None:
        fcf = snapshot.free_cash_flow
        shares = snapshot.shares_outstanding
        if fcf is None or shares in (None, 0):
            return None

        assumptions = self.config["dcf"]
        rate = assumptions["discount_rate"]
        terminal = assumptions["terminal_growth_rate"]
        years = assumptions["forecast_years"]

        if rate <= terminal:
            return None

        projected = sum(fcf / ((1 + rate) ** year) for year in range(1, years + 1))
        terminal_value = fcf * (1 + terminal) / (rate - terminal) / ((1 + rate) ** years)

        return (projected + terminal_value) / shares

    @staticmethod
    def _cagr(values: list[float], years: int) -> float | None:
        if len(values) <= years or values[years] <= 0 or values[0] <= 0:
            return None
        return (values[0] / values[years]) ** (1 / years) - 1


# =========================
# Trading / profile
# =========================       

class ProfileResolver:
    def __init__(self, config: dict[str, Any]) -> None:
        self.config = config

    def classify(self, ticker: str, asset_type: str | None = None) -> Classification:
        override = self.config.get("ticker_classification", {}).get(ticker)
        if override:
            return Classification(market="TW", sector=override["sector"])

        market = "TW"
        sector = "traditional" if asset_type in {"ETF", "MUTUALFUND"} else "technology"
        return Classification(market=market, sector=sector)

    def rules(self, classification: Classification, ticker: str | None = None) -> dict[str, Any]:
        market = self.config["market_profiles"][classification.market]
        sector = self.config["sector_profiles"][classification.sector]
        override = self.config.get("ticker_rule_overrides", {}).get(ticker or "", {})
        return {
            "valuation": market["valuation"] | sector["valuation"] | override.get("valuation", {}),
            "trading": market["trading"] | override.get("trading", {}),
            "quality": sector["quality"] | override.get("quality", {}),
        }


class PriceTargetEngine:
    """
    依股票估值主指標產生價格帶：

    PE 型股票：
        歷史 PE 分位數 × TTM EPS

    PB 型股票：
        歷史 PB 分位數 × BVPS
    """

    @staticmethod
    def _normalized_base(
        snapshot: FinancialSnapshot,
        metric: str,
    ) -> float | None:
        """
        當沒有 Forward EPS Model 時，可作為 PE 型估值的備援基礎。
        PB 型股票直接使用最新 BVPS。
        """
        if metric == "pb":
            return snapshot.book_value_per_share

        # PE 型：優先使用歷史年度 EPS 中位數，
        # 沒有資料則使用目前 EPS。
        values = sorted(
            value
            for value in snapshot.financial_history.get("eps", [])
            if value is not None and value > 0
        )

        if not values:
            return snapshot.ttm_eps or snapshot.eps

        midpoint = len(values) // 2

        if len(values) % 2:
            return float(values[midpoint])

        return float(
            (values[midpoint - 1] + values[midpoint]) / 2
        )

    @staticmethod
    def _get_historical_windows(
        valuation_metrics: ValuationMetrics,
        metric: str,
    ) -> dict[str, Any] | None:
        """Return separate 1Y/3Y/5Y distributions for the primary metric."""
        if metric == "pe":
            return valuation_metrics.historical_pe_window_stats
        if metric == "pb":
            return valuation_metrics.historical_pb_window_stats
        return None

    @staticmethod
    def _get_trailing_base(
        snapshot: FinancialSnapshot,
        metric: str,
    ) -> float | None:
        """
        取得歷史價格帶的每股基礎：

        PE → TTM EPS，若無則 fallback EPS
        PB → 最新 BVPS
        """
        if metric == "pe":
            base = snapshot.ttm_eps or snapshot.eps
        elif metric == "pb":
            base = snapshot.book_value_per_share
        else:
            base = None

        if base is None or base <= 0:
            return None

        return float(base)

    def calculate(
        self,
        snapshot: FinancialSnapshot,
        rules: dict[str, Any],
        valuation_metrics: ValuationMetrics,
    ) -> PriceTargets:
        valuation = rules["valuation"]

        metric = valuation["primary_metric"].lower()
        cheap = valuation[f"{metric}_cheap"]
        expensive = valuation[f"{metric}_expensive"]

        # 不平均1Y/3Y/5Y。通常採3Y；只有三段中樞確認持續升/降評才採1Y。
        historical_windows = self._get_historical_windows(valuation_metrics, metric)
        historical_selection = select_historical_valuation_reference(historical_windows)
        historical_stats = historical_selection["statistics"]

        # 歷史價格帶使用 TTM EPS 或 BVPS
        trailing_base = self._get_trailing_base(
            snapshot,
            metric,
        )

        # PE模型採TTM與前瞻EPS各50%；歷史價格帶仍只採TTM EPS。
        ttm_earnings_base = trailing_base if metric == "pe" else None
        estimated_earnings_base = snapshot.eps_ttm_nowcast if metric == "pe" else None
        if estimated_earnings_base is not None and estimated_earnings_base <= 0:
            estimated_earnings_base = None

        if metric == "pb":
            fair_base = trailing_base
            model_earnings_basis = "BVPS 100%"
        elif ttm_earnings_base is not None and estimated_earnings_base is not None:
            fair_base = 0.50 * ttm_earnings_base + 0.50 * estimated_earnings_base
            model_earnings_basis = "TTM EPS 50% + FinMind即時EPS 50%"
        elif ttm_earnings_base is not None:
            fair_base = ttm_earnings_base
            model_earnings_basis = "TTM EPS 100%（前瞻EPS缺值）"
        elif estimated_earnings_base is not None:
            fair_base = estimated_earnings_base
            model_earnings_basis = "FinMind即時EPS 100%（TTM EPS缺值）"
        else:
            fair_base = None
            model_earnings_basis = "估值基礎資料不足"

            
        # 若主要基礎資料缺失，再用中位數 EPS/BVPS 當 fallback
        if fair_base is None:
            fair_base = self._normalized_base(
                snapshot,
                metric,
            )

        # -------------------------------------------------
        # 決定模型估值倍數
        # 優先使用歷史 P25 / P50 / P75
        # -------------------------------------------------
        if (
            historical_stats is not None
            and historical_stats.get("p25") is not None
            and historical_stats.get("p50") is not None
            and historical_stats.get("p75") is not None
        ):
            cheap_multiple = historical_stats["p25"]
            fair_multiple = historical_stats["p50"]
            expensive_multiple = historical_stats["p75"]
        else:
            # 無足夠歷史資料時，退回 YAML 規則
            cheap_multiple = cheap
            fair_multiple = (cheap + expensive) / 2
            expensive_multiple = expensive

        # -------------------------------------------------
        # 歷史分位數價格帶：P05 / P25 / P50 / P75 / P95
        # PE：TTM EPS × 歷史 PE 分位數
        # PB：BVPS × 歷史 PB 分位數
        # -------------------------------------------------
        historical_P05_price = None
        historical_buy_price = None
        historical_fair_price = None
        historical_sell_price = None
        historical_P95_price = None

        def median_price(period: str) -> float | None:
            window = historical_windows.get(period) if historical_windows else None
            median = window.get("p50") if isinstance(window, dict) else None
            return trailing_base * median if trailing_base is not None and median is not None else None

        historical_median_price_1y = median_price("1y")
        historical_median_price_3y = median_price("3y")
        historical_median_price_5y = median_price("5y")

        required_percentiles = ["p05", "p25", "p50", "p75", "p95"]

        if (
            historical_stats is not None
            and trailing_base is not None
            and all(
                historical_stats.get(key) is not None
                for key in required_percentiles
            )
        ):
            historical_P05_price = (
                trailing_base * historical_stats["p05"]
            )
            historical_buy_price = (
                trailing_base * historical_stats["p25"]
            )
            historical_fair_price = (
                trailing_base * historical_stats["p50"]
            )
            historical_sell_price = (
                trailing_base * historical_stats["p75"]
            )
            historical_P95_price = (
                trailing_base * historical_stats["p95"]
            )

        # -------------------------------------------------
        # 模型價格帶
        # -------------------------------------------------
        ttm_eps_fair_price = (
            ttm_earnings_base * fair_multiple
            if metric == "pe" and ttm_earnings_base is not None
            else None
        )
        estimated_eps_fair_price = (
            estimated_earnings_base * fair_multiple
            if metric == "pe" and estimated_earnings_base is not None
            else None
        )

        model_buy_price = (
            None
            if fair_base is None
            else fair_base * cheap_multiple
        )

        model_fair_price = (
            None
            if fair_base is None
            else fair_base * fair_multiple
        )

        model_sell_price = (
            None
            if fair_base is None
            else fair_base * expensive_multiple
        )

        upside_to_fair = (
            safe_divide(
                model_fair_price - snapshot.current_price,
                snapshot.current_price,
            )
            if (
                model_fair_price is not None
                and snapshot.current_price not in (None, 0)
            )
            else None
        )

        if metric == "pe":
            valuation_applicability = assess_pe_valuation_applicability(
                snapshot.financial_history.get("eps", []),
                snapshot.ttm_eps or snapshot.eps,
            )
        else:
            valuation_applicability = "正常（採PB估值）"

        price_advice = calculate_price_recommendation(
            snapshot.current_price,
            model_fair_price,
            historical_selection["current_percentile"],
            valuation_applicability,
            historical_selection["confidence"],
            historical_selection["reference_period"],
        )
        if metric == "pe" and (ttm_earnings_base is None or estimated_earnings_base is None):
            price_advice["confidence"] = "低"
            price_advice["basis"] += "；TTM／前瞻EPS有一項缺值，未完整套用50/50權重"

        # 說明文字依 PE/PB 改變
        if metric == "pe":
            historical_basis = "historical PE price-band multiple × TTM EPS"
            model_basis = (
                model_earnings_basis
            )
        elif metric == "pb":
            historical_basis = "historical PB price-band multiple × BVPS"
            model_basis = "model uses latest BVPS"
        else:
            historical_basis = "historical valuation price band"
            model_basis = "model valuation"

        return PriceTargets(
            model_buy_price=model_buy_price,
            model_fair_price=model_fair_price,
            model_sell_price=model_sell_price,

            historical_P05_price=historical_P05_price,
            historical_buy_price=historical_buy_price,
            historical_fair_price=historical_fair_price,
            historical_sell_price=historical_sell_price,
            historical_P95_price=historical_P95_price,
            historical_median_price_1y=historical_median_price_1y,
            historical_median_price_3y=historical_median_price_3y,
            historical_median_price_5y=historical_median_price_5y,

            upside_to_fair=upside_to_fair,
            primary_metric=metric.upper(),
            basis=(
                f"{metric.upper()} valuation; {model_basis}; "
                f"historical uses {historical_selection['reference_period'] or 'N/A'} "
                f"{historical_basis}"
            ),
            valuation_environment=historical_selection["environment"],
            historical_reference_period=historical_selection["reference_period"],
            historical_reference_percentile=historical_selection["current_percentile"],
            valuation_applicability=valuation_applicability,
            price_recommendation=price_advice["recommendation"],
            price_recommendation_confidence=price_advice["confidence"],
            price_recommendation_basis=price_advice["basis"],
            valuation_eps_base=fair_base if metric == "pe" else None,
            ttm_eps_fair_price=ttm_eps_fair_price,
            estimated_eps_fair_price=estimated_eps_fair_price,
            model_earnings_basis=model_earnings_basis,
        )
    

In [ ]:
# 舊技術／籌碼引擎定義暫存；基本面批次流程不會呼叫，之後另案拆出。
#技術分析&籌碼 金流分析
# =========================
# TechnicalAnalysis & USFlowProxy 
# =========================
class TechnicalAnalysisEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["technical"]

    def analyze(self, prices: pd.DataFrame) -> TechnicalSignal:
        if prices is None or prices.empty or "Close" not in prices:
            return TechnicalSignal("Unavailable", None, None, None, None, None)

        close = prices["Close"].dropna()
        minimum = max(self.rules["long_ma_days"], self.rules["rsi_days"] + 1)

        if len(close) < minimum:
            return TechnicalSignal("Insufficient Data", None, None, None, None, None)

        short_ma = float(close.tail(self.rules["short_ma_days"]).mean())
        long_ma = float(close.tail(self.rules["long_ma_days"]).mean())

        delta = close.diff()
        gains = delta.clip(lower=0)
        losses = -delta.clip(upper=0)

        avg_gain = gains.ewm(alpha=1 / self.rules["rsi_days"], adjust=False).mean().iloc[-1]
        avg_loss = losses.ewm(alpha=1 / self.rules["rsi_days"], adjust=False).mean().iloc[-1]

        rsi = 100.0 if avg_loss == 0 else float(100 - 100 / (1 + avg_gain / avg_loss))
        support = float(close.tail(self.rules["support_lookback_days"]).min())
        resistance = float(close.tail(self.rules["resistance_lookback_days"]).max())
        latest = float(close.iloc[-1])

        if short_ma > long_ma and rsi <= self.rules["rsi_oversold"]:
            signal = "Buy Setup"
        elif short_ma < long_ma or rsi >= self.rules["rsi_overbought"]:
            signal = "Sell/Reduce Setup"
        elif latest <= support * 1.02:
            signal = "Watch Buy Zone"
        else:
            signal = "Hold/Wait"

        return TechnicalSignal(signal, rsi, short_ma, long_ma, support, resistance)
    


class ChipAnalysisEngine:
    def __init__(self, config: dict[str, Any]) -> None:
        self.rules = config["chip_analysis"]

    def analyze(self, chip_data: pd.DataFrame, entry_price: float | None) -> ChipSignal:
        stop = entry_price * (1 - self.rules["stop_loss_from_entry_pct"]) if entry_price else None
        profit = entry_price * (1 + self.rules["take_profit_from_entry_pct"]) if entry_price else None

        if chip_data is None or chip_data.empty or "institutional_net_buy" not in chip_data:
            return ChipSignal("Unavailable", stop, profit, "Provider has no institutional-flow data")

        flow = chip_data["institutional_net_buy"].tail(self.rules["institutional_buying_days"]).dropna()

        if len(flow) < self.rules["institutional_buying_days"]:
            return ChipSignal("Insufficient Data", stop, profit, "Not enough recent institutional-flow observations")

        signal = "Buy" if (flow > 0).all() else "Sell/Reduce" if (flow < 0).all() else "Hold"
        return ChipSignal(signal, stop, profit, "Recent institutional net-buying trend")


In [ ]:
# 報表產出&中英對照
# =========================
# Report / translations
# =========================
ZH_TW_COLUMNS = {
    "Ticker": "代號",
    "Source": "資料來源",
    "Market": "市場",
    "Sector": "產業",
    "Price": "現價",
    "Price Date": "價格日期",
    "Data Notes": "資料註記",
    "current_pe": "本益比（TTM）",
    "historical_pe_median_1y": "歷史PE中位數_1Y",
    "historical_pe_percentile_1y": "歷史PE百分位_1Y",
    "historical_pe_median_3y": "歷史PE中位數_3Y",
    "historical_pe_percentile_3y": "歷史PE百分位_3Y",
    "historical_pe_median_5y": "歷史PE中位數_5Y",
    "historical_pe_percentile_5y": "歷史PE百分位_5Y",
    "pe_median_trend_1y_vs_3y": "PE中位數趨勢_1Y對3Y",
    "pe_median_trend_1y_vs_5y": "PE中位數趨勢_1Y對5Y",
    "historical_pe_sample_count_1y": "歷史PE有效樣本數_1Y",
    "historical_pe_sample_count_3y": "歷史PE有效樣本數_3Y",
    "historical_pe_sample_count_5y": "歷史PE有效樣本數_5Y",
    "TTM EPS": "近年EPS",
    "EPS TTM Nowcast": "即時EPS",   
    "PE TTM Nowcast": "即時本益比",   
    "Forward EPS": "預估EPS",   
    "Forward EPS Source": "預估EPS來源",
    "Forward EPS Retrieved UTC": "預估EPS取得時間_UTC",
    "Forward EPS Yahoo Symbol": "預估EPS_Yahoo代號",
    "Forward EPS Currency": "預估EPS幣別",
    "Forward EPS Status": "預估EPS資料狀態",
    "Forward PE": "預估本益比",    
    "PB": "股價淨值比",
    "ROIC": "投入資本報酬率",
    "ROE": "股東權益報酬率",
    "FCF Yield": "自由現金流殖利率", #FCF per Share/Price 
    "Revenue Growth 3M": "營收近三月成長率",
    "Revenue Growth 6M": "營收近六月成長率",
    "EPS Growth 3M": "EPS 近三月成長率",
    "EPS Growth 6M": "EPS 近六月成長率",
    "Revenue Growth 3M Status": "營收 3M狀態",
    "Revenue Growth 6M Status": "營收 6M狀態",
    "Revenue Growth 9M": "營收 近九月成長率",
    "Revenue Growth 9M Status": "營收 9M狀態",
    "EPS Growth 3M Status": "EPS 3M狀態",
    "EPS Growth 6M Status": "EPS 6M狀態",
    "EPS Growth 9M": "EPS 近九月成長率",
    "EPS Growth 9M Status": "EPS 9M狀態",
    "OCF Growth 3M": "OCF 近三月成長率",
    "OCF Growth 3M Status": "OCF 3M狀態",
    "OCF Growth 6M": "OCF 近六月成長率",
    "OCF Growth 6M Status": "OCF 6M狀態",
    "OCF Growth 9M": "OCF 近九月成長率",
    "OCF Growth 9M Status": "OCF 9M狀態",
    "Growth 3M Period": "3M比較季度",
    "Growth 6M Period": "6M比較季度",
    "Growth 9M Period": "9M比較季度",
    "OCF Momentum Trend": "OCF逐季趨勢",
    "EPS CAGR 3Y": "EPS 三年複合成長率",
    "EPS Momentum Trend": "EPS短期趨勢",
    "Revenue Momentum Trend": "營收短期趨勢",
    "Valuation Sample Count": "歷史估值樣本數",
    "Valuation History Years": "歷史估值可用年數",
    "Growth Score": "成長分數",
    "Quality Score": "品質分數",
    "Buffett Score": "巴菲特檢核分數",
    "Fundamental Score": "基本面分數",
    "Recommendation": "基本面投資建議",
    "Price Recommendation": "價格投資建議",
    "Valuation Environment": "估值環境",
    "Historical Reference Period": "歷史估值主參考期",
    "Historical Reference Percentile": "主參考期估值百分位",
    "Valuation Applicability": "估值指標適用性",
    "Price Recommendation Confidence": "價格建議信心",
    "Price Recommendation Basis": "價格建議依據",
    "Historical Median Price 1Y": "歷史估值中位價_1Y",
    "Historical Median Price 3Y": "歷史估值中位價_3Y",
    "Historical Median Price 5Y": "歷史估值中位價_5Y",
    "Historical Price Band Reference Period": "歷史價格帶參考期",
    "Valuation EPS Base": "模型綜合EPS",
    "TTM EPS Fair Price": "TTM EPS合理價",
    "Estimated EPS Fair Price": "前瞻EPS合理價",
    "Model Earnings Basis": "模型EPS權重基準",
    "Earnings Recommendation": "盈餘投資建議",
    "Earnings Trend Score": "盈餘趨勢分數",
    "Earnings Recommendation Note": "盈餘建議依據",
    'Reference Notes': '季增率資料註記',
    'CF Financial Currency': '現金流財報幣別',
    'CF TTM Period': '現金流TTM四季期間',
    'OCF TTM': '營業現金流OCF_TTM',
    'Net Income TTM CF Basis': '同期淨利_TTM',
    'Revenue TTM CF Basis': '同期營收_TTM',
    'CapEx TTM Outflow': '資本支出_TTM_正值支出',
    'FCF TTM': '自由現金流FCF_TTM',
    'OCF / Net Income TTM': '盈餘現金轉換率_OCF除以淨利',
    'OCF Quality Status': '盈餘現金轉換狀態',
    'FCF Margin TTM': 'FCF利潤率_TTM',
    'FCF Yield TTM Aligned': 'FCF殖利率_TTM同幣別',
    'FCF Positive Ratio 5Y': 'FCF正值比例_完整五年度',
    'FCF Positive Ratio Available': 'FCF正值比例_可用年度',
    'FCF Positive Years': 'FCF正值年度數',
    'FCF Valid Years': 'FCF有效年度數_最多五年',
    'FCF Positive Streak': 'FCF可確認連續正值年數',
    'FCF Streak Status': 'FCF連續正值說明',
    'Cash Flow Quality Notes': '現金流品質資料註記',
    'Annual FCF 1 Period': 'FCF年度1期末',
    'Annual FCF 1': 'FCF年度1金額',
    'Annual FCF 2 Period': 'FCF年度2期末',
    'Annual FCF 2': 'FCF年度2金額',
    'Annual FCF 3 Period': 'FCF年度3期末',
    'Annual FCF 3': 'FCF年度3金額',
    'Annual FCF 4 Period': 'FCF年度4期末',
    'Annual FCF 4': 'FCF年度4金額',
    'Annual FCF 5 Period': 'FCF年度5期末',
    'Annual FCF 5': 'FCF年度5金額',
    'Cash Quality Review': '現金獲利品質評價',
    'Financial Safety Review': '財務安全性評價',
    "Analyst Consensus": "分析師共識（強買/買進/持有/賣出/強賣）",
    "Analyst Strong Buy Count": "分析師評等_強買家數",
    "Analyst Buy Count": "分析師評等_買進家數",
    "Analyst Hold Count": "分析師評等_持有家數",
    "Analyst Sell Count": "分析師評等_賣出家數",
    "Analyst Strong Sell Count": "分析師評等_強賣家數",
    "Analyst Rating Count": "分析師評等_樣本總數",
    "Analyst Rating Period": "分析師評等_期間",
    "Analyst Target Low": "分析師目標價最低",
    "Analyst Target Mean": "分析師目標價平均",
    "Analyst Target Median": "分析師目標價中位數",
    "Analyst Target High": "分析師目標價最高",
    "Analyst Target Low Upside": "分析師最低目標價上行空間",
    "Analyst Target Mean Upside": "分析師平均目標價上行空間",
    "Analyst Target Median Upside": "分析師中位目標價上行空間",
    "Analyst Target High Upside": "分析師最高目標價上行空間",
    "Analyst Target Currency": "分析師目標價幣別",
    "Analyst Data Source": "分析師資料來源",
    "Analyst Retrieved UTC": "分析師資料取得時間_UTC",
    "Analyst Yahoo Symbol": "分析師資料_Yahoo代號",
    "Analyst Data Status": "分析師資料狀態",
    'Cash Quality Review Note': '現金獲利品質評價依據',
    'Financial Safety Review Note': '財務安全性評價依據',
    'Review Sector': '附加評價_FinMind產業',
    'Review Industry': '附加評價_FinMind子產業',
    'Safety Period': '財務安全_年度財報期末',
    'Safety Source': '財務安全_資料來源',
    'Safety Currency': '財務安全_財報幣別',
    'Safety Debt': '財務安全_年度有息負債',
    'Safety Cash': '財務安全_年度現金',
    'Safety Cash Row': '財務安全_現金採用項目',
    'Safety Operating Income': '財務安全_年度營業利益',
    'Safety Interest': '財務安全_年度利息支出',
    'Safety EBITDA': '財務安全_年度EBITDA',
    'Safety Current Assets': '財務安全_年度流動資產',
    'Safety Current Liabilities': '財務安全_年度流動負債',
    'Safety Net Debt': '財務安全_淨負債',
    'Safety Net Debt EBITDA': '財務安全_淨負債除以EBITDA',
    'Safety Interest Coverage': '財務安全_營業利益除以利息',
    'Safety Current Ratio': '財務安全_流動比率',
    'Review Rule Version': '附加評價_規則版本',
    'Safety Debt Basis': '財務安全_負債組成口徑',
    'Safety EBITDA Source': '財務安全_EBITDA來源',
    'Safety EBITDA Status': '財務安全_EBITDA補值狀態',
    'Safety EBITDA Retrieved UTC': '財務安全_EBITDA取得時間_UTC',
    'Safety EBITDA Yahoo Symbol': '財務安全_EBITDA_Yahoo代號',
    'Safety EBITDA Currency': '財務安全_EBITDA_Yahoo幣別',
    'Safety EBITDA Yahoo Raw': '財務安全_EBITDA_Yahoo原始值',
    'Safety EBITDA Unit Scale': '財務安全_EBITDA_Yahoo除以FinMind單位倍率',
    'Safety EBITDA Unit Checks': '財務安全_EBITDA單位核對',
    'Safety Fallback Status': '財務安全_Yahoo備援狀態',
    'Safety FinMind Status': '財務安全_FinMind原始評價',
    'Safety FinMind Reason': '財務安全_FinMind原始依據',
    'Safety Fallback Reason': '財務安全_Yahoo備援說明',
    'Safety Fallback Retrieved UTC': '財務安全_Yahoo取得時間_UTC',
    'ROIC Source': 'ROIC資料來源',
    'ROIC Status': 'ROIC資料狀態',
    'ROIC Period': 'ROIC_Yahoo財報期末',
    'ROIC Currency': 'ROIC財報幣別',
    'ROIC Retrieved UTC': 'ROIC取得時間_UTC',
    'ROIC Operating Income': 'ROIC採用營業利益',
    'ROIC Equity': 'ROIC採用權益',
    'ROIC Debt': 'ROIC採用有息負債',
    'ROIC Cash': 'ROIC採用現金',
    'ROIC Invested Capital': 'ROIC投入資本',
    'ROIC Tax Assumption': 'ROIC假設稅率',
    'ROIC Formula': 'ROIC計算依據',
    "Stars": "星等",
    "Model Buy Price": "模型預估便宜價",
    "Model Fair Price": "模型預估合理價",
    "Model Sell Price": "模型預估昂貴價",
    "Historical P05 Price": "歷史估值05%價格",
    "Historical Buy Price": "歷史估值25%價格",
    "Historical Fair Price": "歷史估值50%價格",
    "Historical Sell Price": "歷史估值75%價格",
    "Historical P95 Price": "歷史估值95%價格",
    "Fair Value Upside": "合理價上行空間",
    "Target Metric": "目標價依據",
    "Valuation Basis": "估值模型說明",
    "Technical Signal": "技術面訊號",
    "Technical Score": "技術面分數",
    "RSI": "相對強弱指標 RSI",
    "Support": "支撐價",
    "Resistance": "壓力價",
    "Flow Signal": "資金流訊號",
    "Flow Score": "資金流分數",
    "Flow Note": "資金流說明",
    "Error": "錯誤",
}

ZH_TW_VALUES = {
    "technology": "科技",
    "financial": "金融",
    "traditional": "傳產",
    "cyclical": "景氣循環",
    "Strong Buy": "強力買進",
    "Buy": "買進",
    "Hold": "持有",
    "Reduce": "減碼",
    "Avoid": "避開",
    "Insufficient Data": "資料不足",
    "Data Error": "資料錯誤",
    "Buy Setup": "買進訊號形成",
    "Sell/Reduce Setup": "賣出／減碼訊號",
    "Watch Buy Zone": "觀察買點區",
    "Hold/Wait": "持有／等待",
    "Unavailable": "暫無資料",
    "Sell/Reduce": "賣出／減碼",
    "current_pe": "本益比（TTM）",
    "PB": "股價淨值比",
    "Provider has no institutional-flow data": "資料來源未提供法人／籌碼資料",
    "Not enough recent institutional-flow observations": "近期籌碼資料不足",
    "Recent institutional net-buying trend": "近期法人買賣超趨勢",
    "Price/volume history unavailable": "無價格或成交量資料",
    "Not enough price/volume history": "價格或成交量歷史不足",
    "Uptrend with expanding volume": "放量上升趨勢",
    "Downtrend with expanding volume": "放量下跌趨勢",
    "Mixed/neutral flow proxy": "資金流訊號中性",
    "N/A": "無法計算",
}

#格式輸出調整
def format_report(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 先將所有數值欄位保留到小數點後 3 位
    numeric_cols = df.select_dtypes(include=["number"]).columns
    df[numeric_cols] = df[numeric_cols].round(3)

    # 再把評分欄位改成整數
    score_cols = [
        "Growth Score",
        "Quality Score",
        "Buffett Score",
        "Fundamental Score",
        "RSI",
    ]

    for col in score_cols:
        if col in df.columns:
            df[col] = df[col].round(0).astype("Int64")

    return df

class IndependentInvestmentAdvice:
    """獨立參考評價，不回寫基本面分數、星等、定價或其他建議。"""
    TREND_WEIGHTS = {"revenue": 0.25, "eps": 0.35, "ocf": 0.40}
    PERIOD_WEIGHTS = {3: 0.50, 6: 0.30, 9: 0.20}

    @staticmethod
    def _number(value):
        if value is None or isinstance(value, bool):
            return None
        try:
            value = float(value)
            return value if math.isfinite(value) else None
        except (TypeError, ValueError):
            return None

    @staticmethod
    def earnings_label(score):
        return "持續改善" if score >= 80 else "偏向改善" if score >= 60 else "表現分歧" if score >= 40 else "偏向衰退" if score >= 20 else "持續衰退"

    @classmethod
    def earnings(cls, quality, reference):
        # 不以缺漏或負基期當成0分，也不讓剩餘資料自動升權重。
        points = {"成長": 100, "持平": 50, "衰退": 0}
        component_scores, unavailable, special = {}, [], []
        for metric, weight in cls.TREND_WEIGHTS.items():
            component = 0.0
            for months, period_weight in cls.PERIOD_WEIGHTS.items():
                key = f"{metric}_growth_{months}m"
                state = quality.growth_status.get(key, "資料不足")
                rate = cls._number(getattr(quality, key, None))
                if state in {"資料不足", "比率無法計算"}:
                    unavailable.append(key)
                elif state not in points:
                    special.append(f"{key}={state}")
                elif rate is None:
                    unavailable.append(key)
                else:
                    component += points[state] * period_weight
            component_scores[metric] = component
        for metric in ("eps", "ocf"):
            values = reference.get(metric, [])
            latest = cls._number(values[0]) if len(values) else None
            if latest is None:
                unavailable.append(f"{metric}:最新季缺值")
            elif latest <= 0:
                special.append(f"{metric}:最新季非正值")
        if unavailable:
            return {"score": None, "advice": "資料不足", "note": "；".join(unavailable + special)}
        if special:
            return {"score": None, "advice": "盈餘待觀察", "note": "；".join(special)}
        score = sum(component_scores[metric] * weight for metric, weight in cls.TREND_WEIGHTS.items())
        return {"score": score, "advice": cls.earnings_label(score), "note": "營收25%／EPS35%／OCF40%；該季50%／上季30%／上兩季20%。僅評趨勢，非盈餘品質鑑證"}


class FundamentalRiskReviews:
    """初始篩選規則；與基本面評分、買價及盈餘趨勢建議分離。"""
    RULES = {
        "cash_good_conversion": 1.0, "cash_weak_conversion": 0.5,
        "min_fcf_years": 3, "good_fcf_positive_ratio": 0.8,
        "safe_interest_coverage": 5.0, "risk_interest_coverage": 1.0,
        "safe_net_debt_ebitda": 2.0, "risk_net_debt_ebitda": 4.0,
        "safe_current_ratio": 1.0, "max_annual_age_days": 550,
    }
    _number = staticmethod(IndependentInvestmentAdvice._number)

    @classmethod
    def applicability(cls, inputs):
        sector = str(inputs.get("sector") or "").lower()
        industry = str(inputs.get("industry") or "").lower()
        asset = str(inputs.get("asset_type") or "").upper()
        if asset and asset != "EQUITY": return "不適用", "ETF／基金等非一般企業不套用此規則"
        if "financial" in sector or "reit" in industry or any(s in industry for s in ("bank", "insurance", "capital markets", "credit services", "金融", "銀行", "保險", "證券", "不動產投資信託")):
            return "不適用", "金融業或REIT須另用產業專屬口徑"
        if not sector or not asset: return "資料不足", "FinMind產業或資產類型缺漏，無法確認適用性"
        return None, "一般非金融企業初始檢核，非產業相對評分"

    @classmethod
    def cash_quality(cls, cf, inputs):
        status, scope = cls.applicability(inputs)
        if status: return {"label": status, "note": scope}
        try:
            latest = date.fromisoformat(cf["period"].split(" / ")[-1])
            age = (date.today() - latest).days
            if age < 0 or age > 200:
                return {"label":"資料不足","note":"最新現金流季度日期異常或距今超過200天"}
        except (KeyError, TypeError, ValueError, AttributeError):
            return {"label":"資料不足","note":"缺少有效TTM現金流期間"}
        get=lambda key:cls._number(cf.get(key))
        ni,ocf,ratio,margin,years,positive = [get(k) for k in (
            "net_income_ttm","ocf_ttm","ocf_to_net_income","fcf_margin_ttm","fcf_valid_years","fcf_positive_ratio_available")]
        if ni is not None and ni <= 0:
            return {"label":"待觀察","note":"同期淨利非正，不用OCF／淨利評為良好；虧損不等同會計品質有問題"}
        if ocf is not None and ocf <= 0:
            return {"label":"偏弱","note":"TTM營業現金流非正；依已知風險提示，其他項目可能缺漏"}
        if ratio is not None and ratio < cls.RULES["cash_weak_conversion"]:
            return {"label":"偏弱","note":"OCF／同期淨利低於0.5；依已知風險提示，其他項目可能缺漏"}
        if any(v is None for v in (ni,ocf,ratio,margin,years,positive)):
            return {"label":"資料不足","note":"需完整TTM現金流／淨利／營收及年度FCF，不將缺值當零"}
        if years < cls.RULES["min_fcf_years"]:
            return {"label":"資料不足","note":f"可用FCF年度僅{int(years)}年；至少需3年（最多觀察最近5年度）"}
        try:
            annual_age = (date.today() - date.fromisoformat(cf["annual_fcf"][0]["period_end"])).days
            if annual_age < 0 or annual_age > cls.RULES["max_annual_age_days"]:
                return {"label":"資料不足","note":"年度FCF歷史過舊或日期異常"}
        except (KeyError, IndexError, TypeError, ValueError):
            return {"label":"資料不足","note":"年度FCF日期缺漏"}
        if not 0 <= positive <= 1:
            return {"label":"資料不足","note":"FCF正值比例不在有效範圍"}
        findings=[]
        if ratio < cls.RULES["cash_good_conversion"]: findings.append("OCF／同期淨利低於1")
        if margin <= 0: findings.append("FCF Margin非正，需檢視投資支出，並不直接代表盈餘造假")
        if positive < cls.RULES["good_fcf_positive_ratio"]: findings.append("可用年度FCF正值比例低於80%")
        return {"label":"注意" if findings else "良好", "note":"；".join(findings) if findings else f"OCF／淨利≥1、FCF Margin>0、{int(years)}個可用年度FCF正值比例≥80%；僅現金轉換與持續性檢核"}

    @classmethod
    def safety(cls, inputs):
        result={"label":"資料不足","note":"", "net_debt":None,"net_debt_ebitda":None,
                "interest_coverage":None,"current_ratio":None}
        status,scope=cls.applicability(inputs)
        if status:
            return {**result,"label":status,"note":scope}
        try:
            age=(date.today()-date.fromisoformat(inputs["period"])).days
        except (TypeError,ValueError,KeyError):
            return {**result,"note":inputs.get("note") or "缺少可對齊的年度財報日期"}
        if age < 0 or age > cls.RULES["max_annual_age_days"]:
            return {**result,"note":"年度財報日期異常或距今超過550天，不評為穩健"}
        fields=("debt","cash","operating_income","interest","ebitda","current_assets","current_liabilities")
        vals={key:cls._number(inputs.get(key)) for key in fields}
        for key in ("debt","cash","interest","current_assets","current_liabilities"):
            if vals[key] is not None and vals[key]<0: vals[key]=None
        debt,cash,op,interest,ebitda,assets,liab=[vals[k] for k in fields]
        if debt is not None and cash is not None:
            result["net_debt"]=debt-cash
            if ebitda is not None and ebitda>0:result["net_debt_ebitda"]=(debt-cash)/ebitda
        if op is not None and interest is not None and interest>0:result["interest_coverage"]=op/interest
        if assets is not None and liab is not None and liab>0:result["current_ratio"]=assets/liab
        risks=[]
        if result["interest_coverage"] is not None and result["interest_coverage"]<cls.RULES["risk_interest_coverage"]:risks.append("營業利益不足支付利息（低於高風險門檻）")
        if result["net_debt_ebitda"] is not None and result["net_debt_ebitda"]>cls.RULES["risk_net_debt_ebitda"]:risks.append("淨負債／EBITDA超過高風險門檻")
        if result["net_debt"] is not None and result["net_debt"]>0 and ebitda is not None and ebitda<=0:risks.append("淨負債為正且EBITDA非正")
        missing=[key for key,v in vals.items() if v is None]
        if risks:return {**result,"label":"高風險","note":"；".join(risks)+("；另缺漏："+",".join(missing) if missing else "")}
        if missing:return {**result,"note":"年度財報缺漏／數值無效："+",".join(missing)}
        cautions=[]
        if inputs.get("debt_partial"):
            cautions.append("負債僅為已映射借款組成，可能不完整，不評為穩健")
        if ebitda<=0:cautions.append("EBITDA非正，不以負分母判定低槓桿")
        if interest==0:
            if debt>0:cautions.append("有負債但利息為零，無法由保障倍數確認偿債能力")
            elif op<=0:cautions.append("雖無負債但營業利益非正")
        elif result["interest_coverage"]<cls.RULES["safe_interest_coverage"]:cautions.append("利息保障低於穩健門檻")
        if result["net_debt_ebitda"] is not None and result["net_debt_ebitda"]>cls.RULES["safe_net_debt_ebitda"]:cautions.append("淨負債／EBITDA高於穩健門檻")
        if result["current_ratio"] is not None and result["current_ratio"]<cls.RULES["safe_current_ratio"]:cautions.append("流動比率低於穩健門檻")
        return {**result,"label":"注意" if cautions else "穩健", "note":"；".join(cautions) if cautions else "同年度檢核：利息保障≥5倍（或無債且無利息）、淨負債／EBITDA≤2、流動比率≥1（或無流動負債）；不代表未來無風險"}


def order_advice_columns(report):
    """Keep independent advice after Stars and valuation context after analyst status."""
    advice = [
        "Recommendation", "Price Recommendation", "Earnings Recommendation",
        "Cash Quality Review", "Financial Safety Review", "Analyst Consensus",
        "Analyst Target Low", "Analyst Target High",
    ]
    valuation_context = [
        "historical_pe_median_1y", "historical_pe_percentile_1y",
        "historical_pe_median_3y", "historical_pe_percentile_3y",
        "historical_pe_median_5y", "historical_pe_percentile_5y",
        "pe_median_trend_1y_vs_3y", "pe_median_trend_1y_vs_5y",
        "historical_pe_sample_count_1y", "historical_pe_sample_count_3y",
        "historical_pe_sample_count_5y", "Valuation Environment",
        "Historical Reference Period", "Historical Reference Percentile",
        "Valuation Applicability", "Price Recommendation Confidence",
        "Price Recommendation Basis", "Historical Median Price 1Y",
        "Historical Median Price 3Y", "Historical Median Price 5Y",
        "Historical Price Band Reference Period", "Valuation EPS Base",
        "TTM EPS Fair Price", "Estimated EPS Fair Price", "Model Earnings Basis",
    ]
    movable = advice + valuation_context
    columns = [column for column in report.columns if column not in movable]
    if "Stars" not in columns:
        columns.append("Stars")
    advice_index = columns.index("Stars") + 1
    columns[advice_index:advice_index] = [column for column in advice if column in report.columns]
    context_index = (
        columns.index("Analyst Data Status") + 1
        if "Analyst Data Status" in columns
        else len(columns)
    )
    columns[context_index:context_index] = [
        column for column in valuation_context if column in report.columns
    ]
    return report.reindex(columns=columns)


def analyze_universe(
    provider: FinMindTaiwanProvider,
    tickers: list[str],
    config: dict[str, Any],
    language: str = "en",
) -> pd.DataFrame:
    valuation_engine = ValuationEngine(config)
    quality_engine = QualityEngine()
    score_engine = ScoringEngine(config)
    checklist = BuffettChecklist(config)
    decision_engine = DecisionEngine(config)
    profiles = ProfileResolver(config)
    targets = PriceTargetEngine()

    records: list[dict[str, Any]] = []

    for ticker in tickers:
        try:
            snapshot = provider.get_snapshot(ticker)
            prices = provider.get_price_history(ticker)
            historical_fundamentals = provider.get_historical_fundamentals(ticker)

            valuation = valuation_engine.calculate(snapshot, prices, historical_fundamentals)
            quality = quality_engine.calculate(snapshot)

            quality_score = score_engine.quality_score(quality, valuation.fcf_yield)
            growth_score = score_engine.growth_score(quality)
            short_growth_direction = QuarterlyTrendReference.analyze(quality)

            checks = checklist.evaluate(quality, snapshot)
            buffett_score = checklist.score(checks)

            total_score = score_engine.total_score(
                quality_score,
                growth_score,
                buffett_score,
            )

            classification = profiles.classify(ticker, snapshot.asset_type)
            rules = profiles.rules(classification, ticker)

            target = targets.calculate(snapshot, rules, valuation)
            decision = decision_engine.decide(total_score)
            earnings_advice = IndependentInvestmentAdvice.earnings(quality, snapshot.quarterly_reference)
            review_inputs = snapshot.fundamental_review_inputs
            cash_review = FundamentalRiskReviews.cash_quality(snapshot.cashflow_quality, snapshot.cash_review_scope or review_inputs)
            safety_review = FundamentalRiskReviews.safety(review_inputs)
            analyst = provider.get_analyst_consensus(ticker)
            annual_cf = (snapshot.cashflow_quality.get("annual_fcf", []) + [{}] * 5)[:5]


                
            records.append({
                "Ticker": ticker,
                "Source": snapshot.source,
                "Price Date": snapshot.as_of.isoformat(),
                "Data Notes": provider.get_data_notes(ticker),
                "Market": classification.market,
                "Sector": classification.sector,
                "Price": snapshot.current_price,                
                "current_pe": valuation.pe if valuation.pe is not None else "N/A",
                "Forward PE": valuation.forward_pe if valuation.forward_pe is not None else "N/A",
                "historical_pe_median_1y": valuation.historical_pe_median_1y if valuation.historical_pe_median_1y is not None else "N/A",
                "historical_pe_percentile_1y": valuation.historical_pe_percentile_1y if valuation.historical_pe_percentile_1y is not None else "N/A",
                "historical_pe_median_3y": valuation.historical_pe_median_3y if valuation.historical_pe_median_3y is not None else "N/A",
                "historical_pe_percentile_3y": valuation.historical_pe_percentile_3y if valuation.historical_pe_percentile_3y is not None else "N/A",
                "historical_pe_median_5y": valuation.historical_pe_median_5y if valuation.historical_pe_median_5y is not None else "N/A",
                "historical_pe_percentile_5y": valuation.historical_pe_percentile_5y if valuation.historical_pe_percentile_5y is not None else "N/A",
                "pe_median_trend_1y_vs_3y": valuation.pe_median_trend_1y_vs_3y if valuation.pe_median_trend_1y_vs_3y is not None else "N/A",
                "pe_median_trend_1y_vs_5y": valuation.pe_median_trend_1y_vs_5y if valuation.pe_median_trend_1y_vs_5y is not None else "N/A",
                "historical_pe_sample_count_1y": valuation.historical_pe_sample_count_1y,
                "historical_pe_sample_count_3y": valuation.historical_pe_sample_count_3y,
                "historical_pe_sample_count_5y": valuation.historical_pe_sample_count_5y,
                # 已公告最近四季 EPS
                "TTM EPS": snapshot.ttm_eps if snapshot.ttm_eps is not None else "N/A",
                # FinMind 月營收即時推估 EPS
                "EPS TTM Nowcast": snapshot.eps_ttm_nowcast if snapshot.eps_ttm_nowcast is not None else "N/A",
                "PE TTM Nowcast": snapshot.pe_ttm_nowcast if snapshot.pe_ttm_nowcast is not None else "N/A",
                # Yahoo 僅補 Forward EPS；不影響 FinMind EPS 與 Nowcast
                "Forward EPS": snapshot.forward_eps if snapshot.forward_eps is not None else "N/A",
                "Forward EPS Source": snapshot.forward_eps_meta.get("source"),
                "Forward EPS Retrieved UTC": snapshot.forward_eps_meta.get("checked_at"),
                "Forward EPS Yahoo Symbol": snapshot.forward_eps_meta.get("symbol"),
                "Forward EPS Currency": snapshot.forward_eps_meta.get("currency"),
                "Forward EPS Status": snapshot.forward_eps_meta.get("status"),
                "PB": valuation.pb,
                "ROIC": quality.roic,
                "ROE": quality.roe,
                "FCF Yield": valuation.fcf_yield,
                "Revenue Growth 3M": quality.revenue_growth_3m,
                "Revenue Growth 6M": quality.revenue_growth_6m,
                "EPS Growth 3M": quality.eps_growth_3m,
                "EPS Growth 6M": quality.eps_growth_6m,
                "Revenue Growth 3M Status": quality.growth_status["revenue_growth_3m"],
                "Revenue Growth 6M Status": quality.growth_status["revenue_growth_6m"],
                "Revenue Growth 9M": quality.revenue_growth_9m,
                "Revenue Growth 9M Status": quality.growth_status["revenue_growth_9m"],
                "EPS Growth 3M Status": quality.growth_status["eps_growth_3m"],
                "EPS Growth 6M Status": quality.growth_status["eps_growth_6m"],
                "EPS Growth 9M": quality.eps_growth_9m,
                "EPS Growth 9M Status": quality.growth_status["eps_growth_9m"],
                "OCF Growth 3M": quality.ocf_growth_3m,
                "OCF Growth 3M Status": quality.growth_status["ocf_growth_3m"],
                "OCF Growth 6M": quality.ocf_growth_6m,
                "OCF Growth 6M Status": quality.growth_status["ocf_growth_6m"],
                "OCF Growth 9M": quality.ocf_growth_9m,
                "OCF Growth 9M Status": quality.growth_status["ocf_growth_9m"],
                "Growth 3M Period": quality.growth_periods["3m"],
                "Growth 6M Period": quality.growth_periods["6m"],
                "Growth 9M Period": quality.growth_periods["9m"],
                "OCF Momentum Trend": short_growth_direction["ocf_direction"],
                "EPS CAGR 3Y": quality.eps_cagr_3y,
                "EPS Momentum Trend": short_growth_direction["eps_direction"], #額外評價
                "Revenue Momentum Trend": short_growth_direction["revenue_direction"], #額外評價
                "Growth Score": growth_score,
                "Quality Score": quality_score,
                "Buffett Score": buffett_score,
                "Fundamental Score": total_score,
                "Recommendation": decision.label,
                "Stars": decision.stars,
                "Price Recommendation": target.price_recommendation,
                "Earnings Recommendation": earnings_advice["advice"],
                "Model Buy Price": target.model_buy_price,
                "Model Fair Price": target.model_fair_price,
                "Model Sell Price": target.model_sell_price,
                "Historical P05 Price": target.historical_P05_price,
                "Historical Buy Price": target.historical_buy_price,
                "Historical Fair Price": target.historical_fair_price,
                "Historical Sell Price": target.historical_sell_price,
                "Historical P95 Price": target.historical_P95_price,
                "Historical Median Price 1Y": target.historical_median_price_1y,
                "Historical Median Price 3Y": target.historical_median_price_3y,
                "Historical Median Price 5Y": target.historical_median_price_5y,
                "Fair Value Upside": target.upside_to_fair,
                "Target Metric": target.primary_metric,
                "Earnings Trend Score": earnings_advice["score"],
                "Earnings Recommendation Note": earnings_advice["note"],
                "Reference Notes": "; ".join(snapshot.quarterly_reference.get("notes", [])),
                "CF Financial Currency": snapshot.cashflow_quality.get("currency"),
                "CF TTM Period": snapshot.cashflow_quality.get("period"),
                "OCF TTM": snapshot.cashflow_quality.get("ocf_ttm"),
                "Net Income TTM CF Basis": snapshot.cashflow_quality.get("net_income_ttm"),
                "Revenue TTM CF Basis": snapshot.cashflow_quality.get("revenue_ttm"),
                "CapEx TTM Outflow": snapshot.cashflow_quality.get("capex_ttm"),
                "FCF TTM": snapshot.cashflow_quality.get("fcf_ttm"),
                "OCF / Net Income TTM": snapshot.cashflow_quality.get("ocf_to_net_income"),
                "OCF Quality Status": snapshot.cashflow_quality.get("ocf_quality_status"),
                "FCF Margin TTM": snapshot.cashflow_quality.get("fcf_margin_ttm"),
                "FCF Yield TTM Aligned": snapshot.cashflow_quality.get("fcf_yield_ttm"),
                "FCF Positive Ratio 5Y": snapshot.cashflow_quality.get("fcf_positive_ratio_5y"),
                "FCF Positive Ratio Available": snapshot.cashflow_quality.get("fcf_positive_ratio_available"),
                "FCF Positive Years": snapshot.cashflow_quality.get("fcf_positive_years"),
                "FCF Valid Years": snapshot.cashflow_quality.get("fcf_valid_years"),
                "FCF Positive Streak": snapshot.cashflow_quality.get("fcf_positive_streak"),
                "FCF Streak Status": snapshot.cashflow_quality.get("fcf_streak_status"),
                "Cash Flow Quality Notes": "; ".join(snapshot.cashflow_quality.get("notes", [])),
                "Annual FCF 1 Period": annual_cf[0].get("period_end"),
                "Annual FCF 1": annual_cf[0].get("fcf"),
                "Annual FCF 2 Period": annual_cf[1].get("period_end"),
                "Annual FCF 2": annual_cf[1].get("fcf"),
                "Annual FCF 3 Period": annual_cf[2].get("period_end"),
                "Annual FCF 3": annual_cf[2].get("fcf"),
                "Annual FCF 4 Period": annual_cf[3].get("period_end"),
                "Annual FCF 4": annual_cf[3].get("fcf"),
                "Annual FCF 5 Period": annual_cf[4].get("period_end"),
                "Annual FCF 5": annual_cf[4].get("fcf"),
                "Cash Quality Review": cash_review["label"],
                "Financial Safety Review": safety_review["label"],
                "Analyst Consensus": analyst["summary"],
                "Analyst Strong Buy Count": analyst["strong_buy"],
                "Analyst Buy Count": analyst["buy"],
                "Analyst Hold Count": analyst["hold"],
                "Analyst Sell Count": analyst["sell"],
                "Analyst Strong Sell Count": analyst["strong_sell"],
                "Analyst Rating Count": analyst["rating_count"],
                "Analyst Rating Period": analyst["period"],
                "Analyst Target Low": analyst["target_low"],
                "Analyst Target Mean": analyst["target_mean"],
                "Analyst Target Median": analyst["target_median"],
                "Analyst Target High": analyst["target_high"],
                "Analyst Target Low Upside": safe_divide(analyst["target_low"], snapshot.current_price) - 1 if safe_divide(analyst["target_low"], snapshot.current_price) is not None else None,
                "Analyst Target Mean Upside": safe_divide(analyst["target_mean"], snapshot.current_price) - 1 if safe_divide(analyst["target_mean"], snapshot.current_price) is not None else None,
                "Analyst Target Median Upside": safe_divide(analyst["target_median"], snapshot.current_price) - 1 if safe_divide(analyst["target_median"], snapshot.current_price) is not None else None,
                "Analyst Target High Upside": safe_divide(analyst["target_high"], snapshot.current_price) - 1 if safe_divide(analyst["target_high"], snapshot.current_price) is not None else None,
                "Analyst Target Currency": analyst["currency"],
                "Analyst Data Source": analyst["source"],
                "Analyst Retrieved UTC": analyst["checked_at"],
                "Analyst Yahoo Symbol": analyst["symbol"],
                "Analyst Data Status": analyst["status"],
                "Valuation Environment": target.valuation_environment,
                "Historical Reference Period": target.historical_reference_period,
                "Historical Reference Percentile": target.historical_reference_percentile,
                "Valuation Applicability": target.valuation_applicability,
                "Price Recommendation Confidence": target.price_recommendation_confidence,
                "Price Recommendation Basis": target.price_recommendation_basis,
                "Historical Price Band Reference Period": target.historical_reference_period,
                "Valuation EPS Base": target.valuation_eps_base,
                "TTM EPS Fair Price": target.ttm_eps_fair_price,
                "Estimated EPS Fair Price": target.estimated_eps_fair_price,
                "Model Earnings Basis": target.model_earnings_basis,
                "Valuation Basis": target.basis,
                "Cash Quality Review Note": cash_review["note"],
                "Financial Safety Review Note": safety_review["note"],
                "Review Sector": review_inputs.get("sector"),
                "Review Industry": review_inputs.get("industry"),
                "Safety Period": review_inputs.get("period"),
                "Safety Source": review_inputs.get("source"),
                "Safety Currency": review_inputs.get("currency"),
                "Safety Debt": review_inputs.get("debt"),
                "Safety Cash": review_inputs.get("cash"),
                "Safety Cash Row": review_inputs.get("cash_row"),
                "Safety Operating Income": review_inputs.get("operating_income"),
                "Safety Interest": review_inputs.get("interest"),
                "Safety EBITDA": review_inputs.get("ebitda"),
                "Safety Current Assets": review_inputs.get("current_assets"),
                "Safety Current Liabilities": review_inputs.get("current_liabilities"),
                "Safety Net Debt": safety_review["net_debt"],
                "Safety Net Debt EBITDA": safety_review["net_debt_ebitda"],
                "Safety Interest Coverage": safety_review["interest_coverage"],
                "Safety Current Ratio": safety_review["current_ratio"],
                "Review Rule Version": "TW-general-v1",
                "Safety Debt Basis": review_inputs.get("debt_basis"),
                'Safety EBITDA Source': review_inputs.get('ebitda_source'),
                'Safety EBITDA Status': review_inputs.get('ebitda_status'),
                'Safety EBITDA Retrieved UTC': review_inputs.get('ebitda_checked_at'),
                'Safety EBITDA Yahoo Symbol': review_inputs.get('ebitda_yahoo_symbol'),
                'Safety EBITDA Currency': review_inputs.get('ebitda_currency'),
                'Safety EBITDA Yahoo Raw': review_inputs.get('ebitda_yahoo_raw'),
                'Safety EBITDA Unit Scale': review_inputs.get('ebitda_unit_scale'),
                'Safety EBITDA Unit Checks': review_inputs.get('ebitda_unit_checks'),
                'Safety Fallback Status': snapshot.safety_fallback_meta.get("status"),
                'Safety FinMind Status': snapshot.safety_fallback_meta.get("original_status"),
                'Safety FinMind Reason': snapshot.safety_fallback_meta.get("original_note"),
                'Safety Fallback Reason': snapshot.safety_fallback_meta.get("note"),
                'Safety Fallback Retrieved UTC': snapshot.safety_fallback_meta.get("checked_at"),
                'ROIC Source': snapshot.roic_meta.get("source"),
                'ROIC Status': snapshot.roic_meta.get("status"),
                'ROIC Period': snapshot.roic_meta.get("period"),
                'ROIC Currency': snapshot.roic_meta.get("currency"),
                'ROIC Retrieved UTC': snapshot.roic_meta.get("checked_at"),
                'ROIC Operating Income': snapshot.roic_meta.get("operating_income"),
                'ROIC Equity': snapshot.roic_meta.get("equity"),
                'ROIC Debt': snapshot.roic_meta.get("debt"),
                'ROIC Cash': snapshot.roic_meta.get("cash"),
                'ROIC Invested Capital': snapshot.roic_meta.get("invested_capital"),
                'ROIC Tax Assumption': snapshot.roic_meta.get("assumed_tax_rate"),
                'ROIC Formula': snapshot.roic_meta.get("formula"),
                #"Valuation Basis": target.basis,
            })
        except Exception as exc:
            LOGGER.exception("Could not analyze %s", ticker)
            records.append({
                "Ticker": ticker,
                "Source": None,
                "Recommendation": "Data Error",
                "Price Recommendation": "資料不足",
                "Earnings Recommendation": "資料不足",
                "Cash Quality Review": "資料不足",
                "Financial Safety Review": "資料不足",
                "Analyst Consensus": "資料不足",
                "Analyst Data Status": "股票分析失敗",
                "Error": str(exc),
            })

    report = pd.DataFrame(records)
    report = format_report(order_advice_columns(report))

    if language.lower() in {"zh", "zh-tw", "zh_tw"}:
        return report.replace(ZH_TW_VALUES).rename(columns=ZH_TW_COLUMNS)

    return report


# 簡化版欄位：依使用者範例固定名稱、順序；不得插入說明欄。
TW_SIMPLE_COLUMNS = [
    ("Ticker", "代號"),
    ("Sector", "產業"),
    ("Price", "現價"),
    ("current_pe", "本益比（TTM）"),
    ("Forward PE", "預估本益比"),
    ("historical_pe_median_1y", "歷史PE中位數_1Y"),
    ("historical_pe_percentile_1y", "歷史PE百分位_1Y"),
    ("historical_pe_median_3y", "歷史PE中位數_3Y"),
    ("historical_pe_percentile_3y", "歷史PE百分位_3Y"),
    ("historical_pe_median_5y", "歷史PE中位數_5Y"),
    ("historical_pe_percentile_5y", "歷史PE百分位_5Y"),
    ("pe_median_trend_1y_vs_3y", "PE中位數趨勢_1Y對3Y"),
    ("pe_median_trend_1y_vs_5y", "PE中位數趨勢_1Y對5Y"),
    ("Growth Score", "成長分數"),
    ("Quality Score", "品質分數"),
    ("Buffett Score", "巴菲特檢核分數"),
    ("Fundamental Score", "基本面分數"),
    ("Stars", "基本面評價星等"),
    ("Recommendation", "基本面投資評價"),
    ("Price Recommendation", "價格投資建議"),
    ("Earnings Recommendation", "盈餘趨勢評價"),
    ("Cash Quality Review", "現金獲利品質評價"),
    ("Financial Safety Review", "財務安全性評價"),
    ("Analyst Consensus", "分析師共識（強買/買進/持有/賣出/強賣）"),
    ("Analyst Target Low", "分析師目標價最低"),
    ("Analyst Target High", "分析師目標價最高"),
    ("Model Buy Price", "模型預估便宜價"),
    ("Model Fair Price", "模型預估合理價"),
    ("Model Sell Price", "模型預估昂貴價"),
    ("Historical P05 Price", "歷史估值05%價格"),
    ("Historical Buy Price", "歷史估值25%價格"),
    ("Historical Fair Price", "歷史估值50%價格"),
    ("Historical Sell Price", "歷史估值75%價格"),
    ("Historical P95 Price", "歷史估值95%價格"),
    ("Fair Value Upside", "合理價上行空間"),
    ("Target Metric", "目標價依據"),
    ("Revenue Momentum Trend", "營收逐季趨勢"),
    ("EPS Momentum Trend", "EPS逐季趨勢"),
    ("OCF Momentum Trend", "OCF逐季趨勢"),
    ("PE TTM Nowcast", "即時本益比"),
    ("TTM EPS", "近年EPS"),
    ("EPS TTM Nowcast", "即時EPS"),
    ("Forward EPS", "預估EPS"),
    ("PB", "股價淨值比"),
    ("ROIC", "投入資本報酬率"),
    ("ROE", "股東權益報酬率"),
    ("FCF Yield", "自由現金流殖利率"),
    ("Revenue Growth 3M", "該季營收成長率"),
    ("Revenue Growth 6M", "上一季營收成長率"),
    ("Revenue Growth 9M", "上兩季營收成長率"),
    ("EPS Growth 3M", "該季EPS成長率"),
    ("EPS Growth 6M", "上季EPS成長率"),
    ("EPS Growth 9M", "上兩季EPS成長率"),
    ("OCF Growth 3M", "該季OCF成長率"),
    ("OCF Growth 6M", "上季OCF成長率"),
    ("OCF Growth 9M", "上兩季OCF成長率"),
]


def build_simple_report(report: pd.DataFrame) -> pd.DataFrame:
    """同一份分析結果選欄、改名；不重算、不排序、不變更數值或百分比格式。"""
    source = report.copy()
    if "代號" in source.columns:
        source = source.rename(columns={value: key for key, value in ZH_TW_COLUMNS.items()})
    elif "Ticker" in source.columns:
        source = source.replace(ZH_TW_VALUES)
    elif not source.empty:
        raise ValueError("Report must contain Ticker or 代號")
    # 全部股票失敗或清單為空時，仍輸出固定表頭；失敗原因保留在詳細版。
    simple = source.reindex(columns=[key for key, _ in TW_SIMPLE_COLUMNS]).copy()
    simple.columns = [label for _, label in TW_SIMPLE_COLUMNS]
    return simple


CASHFLOW_REPORT_KEYS = ['Ticker', 'Source', 'CF Financial Currency', 'CF TTM Period', 'OCF TTM', 'Net Income TTM CF Basis', 'Revenue TTM CF Basis', 'CapEx TTM Outflow', 'FCF TTM', 'OCF / Net Income TTM', 'OCF Quality Status', 'FCF Margin TTM', 'FCF Yield TTM Aligned', 'FCF Positive Ratio 5Y', 'FCF Positive Ratio Available', 'FCF Positive Years', 'FCF Valid Years', 'FCF Positive Streak', 'FCF Streak Status', 'Cash Flow Quality Notes', 'Annual FCF 1 Period', 'Annual FCF 1', 'Annual FCF 2 Period', 'Annual FCF 2', 'Annual FCF 3 Period', 'Annual FCF 3', 'Annual FCF 4 Period', 'Annual FCF 4', 'Annual FCF 5 Period', 'Annual FCF 5', 'OCF Growth 3M', 'OCF Growth 6M', 'OCF Growth 9M', 'OCF Momentum Trend']

def save_report(report, config, prefix=None):
    """一次分析、兩份 Excel；詳細版與簡化版使用完全獨立的目錄。"""
    from datetime import datetime
    from pathlib import Path
    from openpyxl.styles import Alignment
    from openpyxl.utils import get_column_letter

    config_name = str(config.get("_config_name", "TW"))
    file_prefix = str(prefix or config_name)
    # config 名稱不可變成路徑，確保兩種版本不會因設定而寫到其他位置。
    for name in (config_name, file_prefix):
        if not name or name in {".", ".."} or any(c in name for c in '/\\:<>"|?*'):
            raise ValueError("Report name must be a filename, not a path")
    now = datetime.now()
    root = Path("reports")
    stamp = now.strftime("%Y%m%d_%H%M%S")
    tables = {"詳細版": report.copy(), "簡化版": build_simple_report(report)}
    paths = {}
    for version, table in tables.items():
        output_dir = root / version / config_name
        output_dir.mkdir(parents=True, exist_ok=True)
        filename = output_dir / f"{config_name}_{stamp}_Step1_Report.xlsx"
        with pd.ExcelWriter(filename, engine="openpyxl") as writer:
            table.to_excel(writer, sheet_name="Report", index=False, freeze_panes=(1, 2))
            if version == "詳細版":
                rules = pd.DataFrame([
                    {"規則": key, "設定值": value} for key, value in FundamentalRiskReviews.RULES.items()
                ])
                notes = pd.DataFrame([
                    {"規則": "現金獲利品質", "設定值": "淨利非正待觀察；OCF非正或OCF/淨利<0.5偏弱；資料完整且OCF/淨利≥1、FCF Margin>0、至少3年度且FCF正值比例≥80%為良好，其餘注意"},
                    {"規則": "財務安全性", "設定值": "利息保障<1、淨負債/EBITDA>4、或淨負債正且EBITDA非正為高風險；完整資料達5倍/2倍/流動比率1門檻為穩健，其餘注意"},
                    {"規則": "資料與適用性", "設定值": "台股FinMind優先；財務安全性不足則整組改用Yahoo同年度財報，不混原始金額。FinMind無ROIC時按同公式以Yahoo財報補算，可能連動原有品質分數。只適用一般非金融企業；金融業/REIT/基金不適用；缺值不補零，已知高風險優先提示。安全檢核使用同年度財報，最新TTM季度最多200天、年度最多550天"},
                    {"規則": "ROIC備援", "設定值": "僅FinMind ROIC無法計算時啟用；Yahoo同年度營業利益×80%／(權益＋有息負債−現金)。保留既有20%稅率與期末資本近似；未改用實際稅率或平均投入資本。其他FinMind欄位不覆蓋"},
                    {"規則": "分析師共識與目標價", "設定值": "Yahoo Finance當月（0m）五分類家數：強買/買進/持有/賣出/強賣；目標價保留最低、平均、中位數與最高。簡化版只顯示五分類共識及最低/最高目標價"},
                    {"規則": "分析師資料限制", "設定值": "屬券商分析師彙總共識，不代表台灣投信買賣超；可能缺漏、延遲或受極端目標價影響，只作獨立參考，不納入WHID分數與投資建議"},
                    {"規則": "Historical PE 1Y／3Y／5Y", "設定值": "Current PE以正值TTM EPS計算；歷史PE排除非正值與非有限值。1Y、3Y、5Y須各自具備完整期間覆蓋且至少8筆有效樣本，資料不足顯示N/A且不以短期代替。三期中位數、百分位與中樞趨勢只作估值背景，不納入基本面分數或買賣建議"},
                    {"規則": "限制", "設定值": "初始人工門檻未經回測，不是信用評等或會計品質鑑證；未考慮債務到期分布、授信額度與產業相對門檻；不影響原分數與投資建議"},
                    {"規則": "指標來源（門檻由本模型設定）", "設定值": "https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2026/financial-analysis-techniques"},
                    {"規則": "台股資料來源", "設定值": "https://finmind.github.io/tutor/TaiwanMarket/Fundamental/"},
                    {"規則": "現金品質分析參考", "設定值": "https://www.cfainstitute.org/insights/professional-learning/refresher-readings/2026/evaluating-quality-financial-reports"},
                ])
                pd.concat([notes, rules], ignore_index=True).to_excel(writer, sheet_name="ReviewRules", index=False)
                columns = []
                for key in CASHFLOW_REPORT_KEYS:
                    for candidate in (key, ZH_TW_COLUMNS.get(key, key)):
                        if candidate in table.columns and candidate not in columns:
                            columns.append(candidate)
                if len(columns) > 2:
                    table[columns].to_excel(writer, sheet_name="CashFlowQuality", index=False, freeze_panes=(1, 2))
            for sheet in writer.sheets.values():
                sheet.auto_filter.ref = sheet.dimensions
                sheet.row_dimensions[1].height = 42
                for cell in sheet[1]:
                    label = str(cell.value)
                    width = 29 if "趨勢" in label or "Trend" in label else 18
                    if "註記" in label or "狀態" in label or "Note" in label or "Status" in label:
                        width = 25
                    sheet.column_dimensions[cell.column_letter].width = width
                    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                    if "PE中位數趨勢" in label or "pe_median_trend" in label:
                        for data_cell in sheet.iter_cols(min_col=cell.column, max_col=cell.column, min_row=2):
                            for value_cell in data_cell:
                                if isinstance(value_cell.value, (int, float)):
                                    value_cell.number_format = "0.0%"
                    elif "歷史PE百分位" in label or "historical_pe_percentile_" in label:
                        for data_cell in sheet.iter_cols(min_col=cell.column, max_col=cell.column, min_row=2):
                            for value_cell in data_cell:
                                if isinstance(value_cell.value, (int, float)):
                                    value_cell.number_format = '0.0"%"'
                if sheet.title == "ReviewRules":
                    sheet.column_dimensions["A"].width = 38
                    sheet.column_dimensions["B"].width = 100
                    for row in sheet.iter_rows(min_row=2):
                        for cell in row:
                            cell.alignment = Alignment(vertical="top", wrap_text=True)
                        sheet.row_dimensions[row[0].row].height = 60
        paths[version] = filename
        print(f"{version}已儲存：{filename.resolve()}")
    return paths


In [ ]:
#------------------------------------------------------------------
#Cell 1 — 初始化
#------------------------------------------------------------------
from pathlib import Path
import pandas as pd

# 顯示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# 載入設定與 provider / engines
config = load_config("config/TW.yaml")
provider = FinMindTaiwanProvider(config)

valuation_engine = ValuationEngine(config)
quality_engine = QualityEngine()
score_engine = ScoringEngine(config)
checklist = BuffettChecklist(config)
decision_engine = DecisionEngine(config)
profiles = ProfileResolver(config)
targets = PriceTargetEngine()

print("Initialized.")

In [ ]:
# 台股基本面與估值：一次分析，分開輸出詳細版與簡化版。
# 在方括號內指定股票時只分析指定清單；留空則使用 config/TW.yaml 的 universe.taiwan。
TICKERS = []  # 例如 ["2330.TW", "2449.TW"]；台股請保留 .TW / .TWO 後綴。
configured_tickers = config.get("universe", {}).get("taiwan", [])
tickers = list(TICKERS) if TICKERS else list(configured_tickers)
if not tickers:
    raise ValueError("未指定台股，且 config/TW.yaml 的 universe.taiwan 也是空白")
print("股票清單來源：", "Notebook TICKERS" if TICKERS else "config/TW.yaml")
print("本次評估股票：", tickers)
report = analyze_universe(provider, tickers, config, language="zh-tw")
report_paths = save_report(report, config)
simple_report = build_simple_report(report)
simple_report


In [ ]:
# 選擇性歷史彙整：只讀指定版本，不會混合兩種欄位。
def collect_report_history(version="詳細版"):
    if version not in {"詳細版", "簡化版"}:
        raise ValueError("version must be 詳細版 or 簡化版")
    # ------------------------------------------------------------------
    # Cell — Historical Stock Report Collector (Enhanced)
    # ------------------------------------------------------------------
    from pathlib import Path
    from datetime import datetime
    import pandas as pd

    report_dir = Path("reports") / version

    # 只抓符合命名規則的歷史報表
    files = sorted((report_dir / "TW").glob("TW_*_Step1_Report.xlsx"))      

    all_reports = []

    for file in files:
        try:
            timestamp_str = "_".join(file.stem.split("_")[1:3])
            snapshot_time = datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

            df = pd.read_excel(file)
            if "投資建議" in df.columns and "基本面投資建議" not in df.columns:
                df = df.rename(columns={"投資建議": "基本面投資建議"})

            config_name = file.parent.name

            df.insert(0, "SnapshotTime", snapshot_time)
            df.insert(1, "Config", config_name)
            df.insert(2, "SourceFile", file.name)

            all_reports.append(df)
            print(f"Loaded: {file.name}")

        except Exception as e:
            print(f"Skip {file.name}: {e}")

    # -------------------------------------------------------------
    # 空資料保護
    # -------------------------------------------------------------
    if not all_reports:
        raise ValueError("No report files loaded. Please check the reports folder or file naming pattern.")

    # -------------------------------------------------------------
    # 合併全部歷史報表
    # -------------------------------------------------------------
    history_df = pd.concat(all_reports, ignore_index=True)
    history_df = history_df.sort_values("SnapshotTime").reset_index(drop=True)

    print(f"\nTotal rows: {len(history_df)}")
    if "代號" in history_df.columns:
        print(f"Total tickers: {history_df['代號'].nunique()}")

    # -------------------------------------------------------------
    # 匯出資料夾
    # -------------------------------------------------------------
    history_dir = Path("system_data") / "history" / "TW" / version
    history_dir.mkdir(parents=True, exist_ok=True)

    output_file = history_dir / f"TW_{datetime.now().strftime('%Y%m%d_%H%M%S')}_History.xlsx"

    # -------------------------------------------------------------
    # 最新摘要
    # -------------------------------------------------------------
    if "代號" in history_df.columns:
        latest_df = (
            history_df
            .sort_values("SnapshotTime")
            .groupby("代號", as_index=False)
            .tail(1)
            .sort_values(["Config", "基本面分數"], ascending=[True, False], na_position="last")
            .reset_index(drop=True)
        )
    else:
        latest_df = history_df.copy()

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        # 1. 全部資料總表
        history_df.to_excel(writer, sheet_name="AllReports", index=False)

        # 2. 最新摘要
        latest_df.to_excel(writer, sheet_name="LatestSummary", index=False)

        # 3. 各股票分頁
        if "代號" in history_df.columns:
            tickers = history_df["代號"].dropna().unique()
            used_sheet_names = set(["AllReports", "LatestSummary"])

            for ticker in tickers:
                stock_df = history_df[history_df["代號"] == ticker].copy()
                stock_df = stock_df.sort_values("SnapshotTime").reset_index(drop=True)

                # -------------------------------------------------
                # 新增：現價溢價比例 = 現價 / 基本面買點
                # 新增：現價溢價比例變化
                # -------------------------------------------------
                if "現價" in stock_df.columns and "基本面買點" in stock_df.columns:
                    stock_df["現價溢價比例"] = stock_df["現價"] / stock_df["基本面買點"]
                    stock_df["現價溢價比例變化"] = stock_df["現價溢價比例"].diff()

                # -------------------------------------------------
                # 其他變化欄位
                # -------------------------------------------------
                if "現價" in stock_df.columns:
                    stock_df["現價變化"] = stock_df["現價"].diff()

                if "綜合分數" in stock_df.columns:
                    stock_df["基本面分數變化"] = stock_df["基本面分數"].diff()

                for years in (1, 3, 5):
                    percentile_column = f"歷史PE百分位_{years}Y"
                    if percentile_column in stock_df.columns:
                        stock_df[f"{percentile_column}變化"] = stock_df[percentile_column].diff()

                if "基本面投資建議" in stock_df.columns:
                    stock_df["投資建議變化"] = stock_df["基本面投資建議"].shift(1).fillna("") + " → " + stock_df["基本面投資建議"]
                    stock_df.loc[stock_df.index == 0, "投資建議變化"] = stock_df.loc[stock_df.index == 0, "基本面投資建議"]

                # sheet name（代號即可，避免中文名稱管理）
                base_name = str(ticker)[:31]
                sheet_name = base_name
                counter = 1

                while sheet_name in used_sheet_names:
                    suffix = f"_{counter}"
                    sheet_name = base_name[:31 - len(suffix)] + suffix
                    counter += 1

                used_sheet_names.add(sheet_name)

                stock_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Saved: {output_file}")
    return output_file

# 需要時才手動呼叫：collect_report_history("詳細版")
